In [31]:
# %pip install plotly
# %pip install urllib3
# %pip install kaleido
import io
import os
import requests
import pandas as pd
import numpy as np
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import HTML, clear_output, display
import requests.packages.urllib3.util.connection as urllib3_conn

# Get Data from Supabase

In [32]:


SUPABASE_URL = "https://kocihpxevlowqbguhstf.supabase.co"
SUPABASE_KEY = "sb_publishable_1MWEplxpyp0YOGW_TxZiMQ_HbvtHP5Z"
ALT_CLOUDFLARE_IPS = ["104.16.132.229", "104.18.32.7", "172.67.74.135"]

def _create_censorship_resistant_session(host_domain: str, alt_ip: str) -> requests.Session:
  orig_create_connection = urllib3_conn.create_connection
  def patched_create_connection(address, *args, **kwargs):
    host, port = address
    if host == host_domain:
      host = alt_ip
    return orig_create_connection((host, port), *args, **kwargs)
  session = requests.Session()
  urllib3_conn.create_connection = patched_create_connection
  return session


def functionGetDataFromTable(tableName: str, url: str, key: str, page_size: int = 1000) -> pd.DataFrame:
  endpoint = f"{url.rstrip('/')}/rest/v1/{tableName}?select=*"
  host_domain = url.replace("https://", "").replace("http://", "").split("/")[0]
  headers = {
      "apikey": key.strip(),
      "Authorization": f"Bearer {key.strip()}",
      "Content-Type": "application/json",
  }
  session = None
  for alt_ip in [None] + ALT_CLOUDFLARE_IPS:
    try:
      if alt_ip is None:
        test_session = requests.Session()
      else:
        test_session = _create_censorship_resistant_session(
            host_domain, alt_ip
        )
      check_res = test_session.get(f"{url}/rest/v1/", headers=headers, timeout=5)
      if check_res.status_code < 500:
        session = test_session
        if alt_ip:
          print(f"⚡ Connected via unblocked Cloudflare route: {alt_ip}")
        break
    except requests.exceptions.RequestException:
      continue

  if session is None:
    print("❌ Failed to reach Supabase across all routes. ISP block may require a Cloudflare Worker Relay.")
    return None

  all_data = []
  start_index = 0

  try:
    while True:
      page_headers = headers.copy()
      page_headers["Range"] = f"{start_index}-{start_index + page_size - 1}"
      response = session.get(endpoint, headers=page_headers, timeout=15)
      response.raise_for_status()
      chunk = response.json()
      if not chunk:
        break
      all_data.extend(chunk)
      if len(chunk) < page_size:
        break
      start_index += page_size

    print(f"✅ Successfully retrieved ALL {len(all_data)} rows from '{tableName}'")
    return pd.DataFrame(all_data)

  except Exception as err:
    print(f"❌ An error occurred during data retrieval: {err}")
    return None

In [33]:
df_ygntbpro_supabase = functionGetDataFromTable("ygntbpro",SUPABASE_URL,SUPABASE_KEY)
df_ygntbpro_supabase

✅ Successfully retrieved ALL 33316 rows from 'ygntbpro'


,Date,Team,StateRegionName,Tsp,WardVillage,WardCode,Approach,Visit_no,Sr_No,ReportingPeriod,...,Date_drug_pickup_6,Date_drug_pickup_7,Date_drug_pickup_8,Date_drug_pickup_9,TPTdiscontinuationdate,TPTTreatmentOutcome,OutcomeDate,Recstatus,TypeofTBTreatment,UniqueKey
0,2026-01-05,1,None,MYG,4 WARD,MMR013042701504,PPM,1,1,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN
1,2026-06-15,5,None,HTY,Industrial Zone(HTY-HUADA(Myanmar) Factory),MMR013047701504,Mobile Visit,18,109,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN
2,2026-01-06,1,Yangon,MYG,4 WARD,MMR013042701504,PPM,1,19,1st Qtr,...,None,None,None,None,None,None,None,NaN,1.0,NaN
3,2026-01-05,1,None,MYG,4 WARD,MMR013042701504,PPM,1,3,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN
4,2026-01-05,1,None,MYG,4 WARD,MMR013042701504,PPM,1,4,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33311,2026-06-22,5,None,HTY,Industrial Zone(HTY-DIHUALI(Myanmar) Factory,MMR013047701504,Mobile Visit,19,460,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN
33312,2026-06-22,5,None,HTY,Industrial Zone(HTY-DIHUALI(Myanmar) Factory,MMR013047701504,Mobile Visit,19,461,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN
33313,2026-06-22,5,None,HTY,Industrial Zone(HTY-DIHUALI(Myanmar) Factory,MMR013047701504,Mobile Visit,19,462,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN
33314,2026-06-22,5,None,HTY,Industrial Zone(HTY-DIHUALI(Myanmar) Factory,MMR013047701504,Mobile Visit,19,463,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN


In [34]:
df_target_supabase = functionGetDataFromTable("target",SUPABASE_URL,SUPABASE_KEY)
df_target_supabase

✅ Successfully retrieved ALL 684 rows from 'target'


,ReportingDate,Team,Tsp,Indicator,TargetCategory,Group,Target,PatientID
0,2026-02-25,5,KMD,BC Cases,PPM,,9.00,25-Feb-2026_5_KMD_BC Cases_PPM_
1,2026-03-25,5,KMD,BC Cases,PPM,,9.00,25-Mar-2026_5_KMD_BC Cases_PPM_
2,2026-04-25,5,KMD,BC Cases,PPM,,9.00,25-Apr-2026_5_KMD_BC Cases_PPM_
3,2026-05-25,5,KMD,BC Cases,PPM,,9.00,25-May-2026_5_KMD_BC Cases_PPM_
4,2026-06-25,5,KMD,BC Cases,PPM,,9.00,25-Jun-2026_5_KMD_BC Cases_PPM_
...,...,...,...,...,...,...,...,...
679,2026-05-25,1,HTY,Notified Cases,Mobile,,21.25,25-May-2026_1_HTY_Notified Cases_Mobile_
680,2026-06-25,1,HTY,Notified Cases,Mobile,,21.25,25-Jun-2026_1_HTY_Notified Cases_Mobile_
681,2026-07-25,1,HTY,Notified Cases,Mobile,,21.25,25-Jul-2026_1_HTY_Notified Cases_Mobile_
682,2026-08-25,1,HTY,Notified Cases,Mobile,,21.25,25-Aug-2026_1_HTY_Notified Cases_Mobile_


# Selecting Dataframes

In [35]:
df_dashboard = df_ygntbpro_supabase.copy()
df_target = df_target_supabase.copy()

# Preparation Data

In [36]:
COLUMN_UNCODE = ['Team','Sex','VOL','Referralfor','Cough','Fever','Wtloss','Nightsweat','Haemoptysis','Chestpain','Fatigue','Neckglands',
              'TBcontact','MDRTBcontact','TBTreatmenthistory','Smoking','Reasonforexamination','TypeofPatient','PublicHealthCare1',
              'TypeofPatient1','DM1','HT1','DMHT1','RTIAVI1','Generalweakness1','Other1','Cxrr','CXRresult','Sputum_request','Micror',
              'Sputummicroscopyresult','Genexpertrequested','GeneXpertresult','Bact_status','Case','Treatmentreferral','TreatmentRegimen',
              'Placeforreferral','TreatmentOutcome1211','ContactInvestigation111','DOTSupervision111','DOTsupervisiontillTreatmentComp111',
              'Seeing1','Hearing1','Walking1','Cognition1','Selfcare1','Communication1','Disability1','Xray2ndReading11','CXRresult211',
              'TypeofTBTreatment']

COLUMN_DISABILITY = ["Seeing1","Hearing1","Walking1","Cognition1","Selfcare1","Communication1"]

COLUMN_SYMPTOM = ['Cough','Fever','Wtloss','Nightsweat','Haemoptysis','Chestpain','Fatigue','Neckglands']

COLUMN_PRESERVED_FOR_TARGET = ["ReportingDate","Team","Tsp","TargetCategory","Group"]

UNCODE_DISABILITY = {"1": "No - No Difficulty","2": "Yes - Some Difficulty","3": "Yes - A lot of Difficulty","4": "Yes - Can not do it at all"}

UNCODE_MAPPING = {
    "CXRresult": {
        "1": "Normal",
        "2": "TB Active",
        "3": "TB Suspect",
        "4": "TB Healed",
        "5": "Other Abnormal",
    },
    "CXRresult211": {
        "1": "Normal",
        "2": "TB Active",
        "3": "TB Suspect",
        "4": "TB Healed",
        "5": "Other Abnormal",
    },
    "GeneXpertresult": {
        "0": "N",
        "1": "I",
        "2": "T",
        "3": "RR",
        "4": "TI",
        "5": "Denied",
        "6": "Missing",
        "7": "TT",
    },
    "Placeforreferral": {
        "1": "NTP",
        "2": "MMA",
        "3": "PSI",
        "4": "MATA",
        "5": "Other",
    },
    "TreatmentRegimen": {
        "1": "IR",
        "2": "RR",
        "3": "CR",
        "4": "MDR",
        "5": "MR",
    },
    "TypeofTBTreatment": {"1": "DS-TB", "2": "DR-TB", "3": "TPT"},
    "Sex": {"1": "Male", "2": "Female"},
    "Cxrr": {"1": "Requested", "2": "Not Requested"},
    "Reasonforexamination": {"1": "Diagnosis", "2": "Follow-Up"},
    "VOL": {"1": "Volunteer Referral", "2": "Walk-In"},
    "Referralfor": {"1": "Presumptive", "2": "CI"},
    "Case": {"1": "TB", "2": "No TB"},
    "DM1": {"1": "DM-New", "2": "No DM", "3": "DM-Old"},
    "HT1": {"1": "HT-New", "2": "No DM", "3": "HT-Old"},
    "HIVStatus": {"N": "Negative", "P": "Positive", "U": "Unknown"},
    "Genexpertrequested":{"1":"Requested","2":"Not Requested"},
    "Bact_status": {"1": "BC","2": "CD"},
    "Treatmentreferral": {"1": "Registered", "2": "Not Registered"},
    "TypeofPatient1":{"1":"New","2":"Old"},
    "Team":{"1":"MMA", "5":"MATA"}}

UNCODE_DEFAULT = {"1": "Yes", "2": "No"}

MISSING_STRINGS = {"","none","nan","null","n/a","na","<na>","nat","#n/a","-","None","NONE","NaN","NULL","<NA>","N/A","NaT"}

CRITERIA_INDICATORS = {"Examined Cases": {"Reasonforexamination": "Diagnosis"},
                       "Notified Cases": {"Reasonforexamination": "Diagnosis", "Case": "TB"},
                       "BC Cases": {"Reasonforexamination": "Diagnosis","Case": "TB","Bact_status": "BC"}}

CATEGORY_PHC_CRITERIA = {"DM1": {"DM-New": "DM","DM-Old": "DM"},
                         "HT1": {"HT-New": "HT","HT-Old": "HT"},
                         'RTIAVI1':{"Yes":"AVI"}, 
                         'Generalweakness1':{"Yes":"General Weakness"},
                         'Other1':{"Yes":"Others"}}

COLUMN_CI_DOTS = ['Case','Bact_status','Treatmentreferral','TypeofTBTreatment','Age',
                  'HIVStatus','ContactInvestigation111', 'DOTSupervision111', 
                  'DOTStartedDate111','DOTsupervisiontillTreatmentComp111',
                  'Tsp','Ptstsp','VOL','Referralfor','VolunteerName','Organization',
                  'TreatmentOutcome1211','Tx_Outcome_Date','DOTvolName111',
                  'VolunteerGender111','VolunteerOrganization111']

In [37]:
def normalize(val):
  if pd.isna(val):
    return ""
  s = str(val).strip()
  if s.lower() in MISSING_STRINGS:
    return ""
  try:
    f = float(s)
    return str(int(f)) if f.is_integer() else str(f)
  except (ValueError, TypeError):
    return s
      
def clean_missing(df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()
  non_datetime_cols = [col for col in df.columns if not pd.api.types.is_datetime64_any_dtype(df[col])]
  for col in non_datetime_cols:
    df[col] = df[col].astype(str).str.strip()
  df[non_datetime_cols] = df[non_datetime_cols].replace({s: "" for s in MISSING_STRINGS})
  return df

def function_uncode(df: pd.DataFrame, colName=None, mapping=None) -> pd.DataFrame:
  df = clean_missing(df)
  mapping = mapping or {}
  if colName is None:
    print("No columns specified for uncode. Please provide a column name or list of column names.")
    return df
  elif isinstance(colName, str):
    columns = [colName]
  else:
    columns = list(colName)
  for col in columns:
    if (col not in df.columns or pd.api.types.is_datetime64_any_dtype(df[col])):
      continue
    if col in mapping:
      mp = {normalize(k): v for k, v in mapping[col].items()}
    elif col in COLUMN_DISABILITY:
      mp = UNCODE_DISABILITY
    else:
      mp = UNCODE_DEFAULT
    df[col] = df[col].apply(lambda x: mp.get(normalize(x), x if x else ""))
  return df

def switchingRowToColumn(df, column_name, preserved_column_list, value_col=None):
    if value_col:
        df_reshaped = df.pivot_table(
            index=preserved_column_list,
            columns=column_name,
            values=value_col,
            aggfunc="first",
        ).reset_index()
    else:
        df_reshaped = (
            df.groupby(preserved_column_list + [column_name])
            .size()
            .unstack(fill_value=0)
            .reset_index()
        )
    df_reshaped.columns.name = None
    return df_reshaped

def function_reporting_period(df,date_col="Date",cutoff=25):
    df = df.copy()
    d = pd.to_datetime(df[date_col])
    df["ReportingDate"] = np.where(d.dt.day > cutoff,(d + pd.DateOffset(months=1)).dt.to_period("M").dt.to_timestamp(),d.dt.to_period("M").dt.to_timestamp())
    return df

def create_category(df,source_col,criteria_mapping,output_col="COLUMN_NEW",default=""):
    df = df.copy()
    source = df[source_col]
    conditions = [source.isin(source_values) for source_values in criteria_mapping.values()]
    choices = list(criteria_mapping.keys())
    df[output_col] = np.select(conditions,choices,default=default)
    return df

def create_category_combined(df, criteria_dict, new_column_name='PHC Category', sep=', '):
    df_copy = df.copy()
    mapped_columns = [
        df_copy[col].map(mapping) 
        for col, mapping in criteria_dict.items() 
        if col in df_copy.columns]
    if mapped_columns:
        combined_df = pd.concat(mapped_columns, axis=1)
        df_copy[new_column_name] = combined_df.apply(
            lambda row: sep.join(dict.fromkeys(row.dropna().astype(str))), axis=1
        ).replace('', np.nan)
    else:
        df_copy[new_column_name] = np.nan
    return df_copy

def ci_entitled(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()
    base_filter = (df["Case"] == "TB") & (df["Treatmentreferral"] == "Registered")
    age_numeric = pd.to_numeric(df["Age"], errors="coerce")
    cond_dr_tb = base_filter & (df["TypeofTBTreatment"] == "DR-TB")
    cond_tb_hiv = base_filter & (df["HIVStatus"] == "P")
    cond_under5 = base_filter & (age_numeric < 5)
    cond_dstb_bc = base_filter & (df["Bact_status"] == "BC")
    conditions = [cond_dr_tb, cond_tb_hiv, cond_under5, cond_dstb_bc]
    choices = ["DR-TB", "TB-HIV", "Under5", "DS-TB_BC"]
    df["ECI"] = np.select(conditions, choices, default=None)
    return df

In [38]:
df_target = switchingRowToColumn(df=df_target, column_name="Indicator",preserved_column_list=COLUMN_PRESERVED_FOR_TARGET,value_col="Target")
df_target = function_uncode(df=df_target,colName=["Team"], mapping=UNCODE_MAPPING)
df_target = function_reporting_period(df_target,date_col="ReportingDate")
df_target.rename(columns={"Group":"Clinic"},inplace=True)


In [39]:

mapping_TargetCategory = {"PPM": ["PPM", "Diagnostic Center"],"Mobile": ["Mobile Visit", "Elderly Care", "Touring"]}
df_dashboard = create_category(df_dashboard,source_col="Approach",criteria_mapping=mapping_TargetCategory,output_col="TargetCategory",default="")
df_dashboard.rename(columns={"EPI11":"Clinic"},inplace=True)
df_dashboard = function_uncode(df_dashboard,colName=COLUMN_UNCODE, mapping=UNCODE_MAPPING)
df_dashboard = function_reporting_period(df_dashboard)
df_dashboard = create_category_combined(df_dashboard,CATEGORY_PHC_CRITERIA,"PrimaryHealthcare")



In [62]:
achievement = function_indicator_achievement(df_dashboard,CRITERIA_INDICATORS)
achievement

,ReportingDate,Team,TargetCategory,Tsp,Clinic,Examined Cases,Notified Cases,BC Cases
0,2026-01-01,MATA,Mobile,HLG,,352,12,3
1,2026-01-01,MATA,Mobile,HTY,,754,7,2
2,2026-01-01,MATA,PPM,HLG,,151,6,4
3,2026-01-01,MATA,PPM,KMD,,216,23,17
4,2026-01-01,MATA,PPM,SDG,107,130,11,6
...,...,...,...,...,...,...,...,...
80,2026-07-01,MATA,PPM,HLG,,225,4,4
81,2026-07-01,MATA,PPM,KMD,,166,16,13
82,2026-07-01,MATA,PPM,SDG,107,93,6,5
83,2026-07-01,MATA,PPM,SDG,19,136,13,9


In [63]:
progress = function_merge_target(achievement,df_target,indicators=tuple(CRITERIA_INDICATORS.keys()))
progress

,ReportingDate,Team,TargetCategory,Tsp,Clinic,Examined Cases Target,Notified Cases Target,BC Cases Target,Examined Cases Achievement,Notified Cases Achievement,BC Cases Achievement
0,2026-01-01,MMA,Mobile,HTY,,960.0000,21.25000,8.41667,597.0,2.0,1.0
1,2026-01-01,MMA,PPM,MYG,,113.0000,11.33330,6.75000,184.0,24.0,19.0
2,2026-01-01,MMA,PPM,SOK,,185.7500,18.58330,11.16670,204.0,13.0,9.0
3,2026-01-01,MMA,PPM,SPT,,131.7500,13.16670,7.91667,77.0,7.0,5.0
4,2026-01-01,MMA,PPM,TGG,,137.2500,13.75000,8.25000,126.0,9.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...
139,2026-12-01,MATA,Mobile,HTY,,640.0000,14.00000,5.66667,NaN,NaN,NaN
140,2026-12-01,MATA,PPM,KMD,,150.2500,15.00000,9.00000,NaN,NaN,NaN
141,2026-12-01,MATA,PPM,SDG,107,56.6667,5.66667,3.41667,NaN,NaN,NaN
142,2026-12-01,MATA,PPM,SDG,19,113.2500,11.33330,6.75000,NaN,NaN,NaN


# Plotly Functions

Export Chart

In [40]:

def export_chart(
    fig: go.Figure,
    fig_name: str,
    width: int = 1200,
    height: int = 700,
    scale: float = 3.0,
):
    """
    Exports a Plotly figure to a high-resolution static image or vector file.
        Parameters:
            -----------
            fig : go.Figure
                The Plotly figure object to export.
            fig_name : str
                Target file path/name including extension (e.g., 'chart.png',
                'chart.svg', 'chart.pdf').
            width : int, default=1200
                Base image width in pixels.
            height : int, default=700
                Base image height in pixels.
            scale : float, default=3.0
                Resolution multiplier for raster formats (PNG, JPEG). Scale 3.0 on
                1200x700 yields 3600x2100 px (~300+ DPI).
    """
    
    if not isinstance(fig, go.Figure):
        raise TypeError(
            "The 'fig' parameter must be a Plotly go.Figure object."
        )

    _, ext = os.path.splitext(fig_name)
    ext = ext.lower()

    if not ext:
        raise ValueError(
            "fig_name must include a valid file extension (e.g., '.png', '.svg', '.pdf')."
        )

    try:
        if ext in [".svg", ".pdf"]:
            pio.write_image(fig, fig_name, width=width, height=height)
            print(f"Vector graphic exported successfully: {fig_name}")
        else:
            pio.write_image(
                fig, fig_name, width=width, height=height, scale=scale
            )
            print(
                f"High-res image exported successfully ({int(width*scale)}x{int(height*scale)} px): {fig_name}"
            )
    except ValueError as e:
        if "kaleido" in str(e).lower():
            print(
                "\n[ERROR] Kaleido library is missing. Install it by running:\n"
                "  !pip install -U kaleido\n"
                "Then restart your Jupyter kernel and try again.\n"
            )
        else:
            raise e


# # 1. Export High-Res PNG for Presentations/Web (3600 x 2100 px @ 3x scale)
# export_chart(fig, "chart_high_res.png")

# # 2. Export Lossless Vector PDF for Reports/Publications
# export_chart(fig, "chart_report.pdf")

# # 3. Export SVG for Vector Graphic Software (Illustrator, Inkscape)
# export_chart(fig, "chart_vector.svg")

# # 4. Custom Dimensions & Ultra-HD Resolution (4x scale)
# export_chart(fig, "banner_chart.png", width=1600, height=900, scale=4.0)
# ```<ElicitationsGroup message="Next steps to consider:">

#   <Elicitation label="Add interactive HTML export support" query="Add support for saving dynamic, interactive HTML chart files to the export_chart function."/>

#   <Elicitation label="Integrate export directly into plotting function" query="Show how to integrate export_chart cleanly back into the main plotly_combo_bar_percent function."/>
# </ElicitationsGroup>

Plotly_TargetAchievement

In [41]:
def function_indicator_achievement(dataframe,criteria_indicator,group_columns=None):
    df = dataframe.copy()
    if group_columns is None:
        group_columns = ["ReportingDate","Team","TargetCategory","Tsp","Clinic"]

    for indicator, rules in criteria_indicator.items():
        flag = pd.Series(True, index=df.index)
        for column, value in rules.items():
            if column not in df.columns:
                raise KeyError(
                    f"Column '{column}' required for "
                    f"indicator '{indicator}' was not found.")
            flag &= (
                df[column]
                .fillna("")
                .astype(str)
                .str.strip()
                .eq(str(value).strip()))
        df[indicator] = flag.astype(int)
    indicator_columns = list(criteria_indicator.keys())
    summary = (df.groupby(group_columns, as_index=False)[indicator_columns].sum())
    return summary

def function_merge_target(achievement,target,indicators=("Examined Cases", "Notified Cases", "BC Cases")):
    keys = ["ReportingDate", "Team", "TargetCategory", "Tsp", "Clinic"]
    indicators = list(indicators)
    missing_ach = [c for c in keys + indicators if c not in achievement.columns]
    missing_tar = [c for c in keys + indicators if c not in target.columns]
    if missing_ach:
        raise KeyError(f"Missing columns in achievement: {missing_ach}")
    if missing_tar:
        raise KeyError(f"Missing columns in target: {missing_tar}")
    ach = achievement[keys + indicators].copy()
    tar = target[keys + indicators].copy()
    ach["ReportingDate"] = pd.to_datetime(ach["ReportingDate"], errors="coerce")
    tar["ReportingDate"] = pd.to_datetime(tar["ReportingDate"], errors="coerce")
    min_year = ach["ReportingDate"].dt.year.min()
    max_year = ach["ReportingDate"].dt.year.max()
    tar = tar[tar["ReportingDate"].dt.year.between(min_year, max_year)].copy()
    for col in indicators:
        # tar[col] = (pd.to_numeric(tar[col], errors="coerce").round().astype("Int64"))
        # ach[col] = (pd.to_numeric(ach[col], errors="coerce").round().astype("Int64"))
        tar[col] = (pd.to_numeric(tar[col], errors="coerce"))
        ach[col] = (pd.to_numeric(ach[col], errors="coerce"))
    tar = tar.rename(columns={c: f"{c} Target" for c in indicators})
    ach = ach.rename(columns={c: f"{c} Achievement" for c in indicators})
    return tar.merge(ach, on=keys, how="left")

# achievement = function_indicator_achievement(df_filtered,CRITERIA_INDICATORS)
# progress = function_merge_target(achievement,df_target,indicators=tuple(CRITERIA_INDICATORS.keys()))


def plotly_achievement_target_dropdown(
    dataframe: pd.DataFrame,
    achievement_columnList: list,
    target_columnList: list,
    period: str = "Monthly",
    date_col: str = "Date",
) -> go.Figure:
    """Grouped bar chart on a log scale using native add_hline annotations

    positioned on the top-right of each target line.
    """
    df_clean = dataframe.copy()
    df_clean[date_col] = pd.to_datetime(df_clean[date_col])
    df_clean["Year"] = df_clean[date_col].dt.year

    periods_config = {
        "Monthly": {
            "label": "Monthly",
            "freq": "MS",
            "date_fmt": "%b %Y",
            "divisor": 12,
        },
        "Quarterly": {
            "label": "Quarterly",
            "freq": "QS",
            "date_fmt": "Q%q %Y",
            "divisor": 4,
        },
        "Semiannually": {
            "label": "Semiannually",
            "freq": "6MS",
            "date_fmt": "%b %Y",
            "divisor": 2,
        },
        "Annually": {
            "label": "Annually",
            "freq": "YS",
            "date_fmt": "%Y",
            "divisor": 1,
        },
    }

    # Color palette shared between bars and horizontal target lines
    shared_colors = ["#2ca02c", "#ff7f0e", "#d9381e", "#9467bd", "#17becf"]

    annual_targets = df_clean.groupby("Year")[target_columnList].sum()
    frames_data = {}

    for p_key, p_cfg in periods_config.items():
        agg_df = (
            df_clean.set_index(date_col)
            .resample(p_cfg["freq"])[achievement_columnList]
            .sum()
            .reset_index()
        )
        agg_df["Year"] = agg_df[date_col].dt.year

        if p_key == "Quarterly":
            agg_df["Period_Label"] = agg_df[date_col].dt.to_period(
                "Q"
            ).astype(str)
        elif p_key == "Semiannually":
            # Inside periods_config for Semiannually
            agg_df["Period_Label"] = agg_df[date_col].dt.year.astype(str) + "S" + (agg_df[date_col].dt.month.gt(6).astype(int) + 1).astype(str)
        else:
            agg_df["Period_Label"] = agg_df[date_col].dt.strftime(
                p_cfg["date_fmt"]
            )

        period_targets = {}
        for t_col in target_columnList:
            yearly_t = agg_df["Year"].map(annual_targets[t_col])
            period_targets[t_col] = (yearly_t / p_cfg["divisor"]).mean()

        frames_data[p_key] = {
            "agg_df": agg_df,
            "period_targets": period_targets,
        }

    def get_log_axis_config(agg_df, period_targets):
        target_vals = list(period_targets.values())
        bar_vals = agg_df[achievement_columnList].values.flatten()
        all_vals = [v for v in np.append(bar_vals, target_vals) if v > 0]

        if not all_vals:
            min_exp, max_exp = 0, 4
        else:
            min_val, max_val = min(all_vals), max(all_vals)
            min_exp = int(np.floor(np.log10(min_val)))
            max_exp = int(np.ceil(np.log10(max_val * 1.3)))

        power_ticks = [10**i for i in range(min_exp, max_exp + 1)]
        power_texts = [f"{v:,.0f}" for v in power_ticks]

        return dict(
            type="log",
            tickmode="array",
            tickvals=power_ticks,
            ticktext=power_texts,
            range=[min_exp - 0.2, max_exp],
            gridcolor="#e5e5e5",
            side="left",
        )

    def build_chart_elements(selected_period):
        p_data = frames_data[selected_period]
        agg_df = p_data["agg_df"]
        p_targets = p_data["period_targets"]

        traces = []
        for i, ach_col in enumerate(achievement_columnList):
            color = shared_colors[i % len(shared_colors)]
            traces.append(
                go.Bar(
                    x=agg_df["Period_Label"],
                    y=agg_df[ach_col],
                    name=ach_col,
                    marker_color=color,
                    text=agg_df[ach_col],
                    texttemplate="%{text:,.0f}",
                    textposition="inside", # option :"auto", "inside", "outside"
                    insidetextanchor="middle", # DELETE OR ADD (option : "end" , "start" , "middle"
                    hovertemplate=f"<b>%{{x}}</b><br>{ach_col}: %{{y:,.2f}}<extra></extra>",
                )
            )

        shapes = []
        annotations = []
        for j, t_col in enumerate(target_columnList):
            t_val = p_targets[t_col]
            color = shared_colors[j % len(shared_colors)]

            # Native line shape across plot frame
            shapes.append(
                dict(
                    type="line",
                    xref="paper",
                    x0=0,
                    x1=1,
                    yref="y",
                    y0=t_val,
                    y1=t_val,
                    line=dict(color=color, width=2.5, dash="dash"),
                )
            )

            # Target annotation placed inside top-right of the plot line
            annotations.append(
                dict(
                    xref="paper",
                    x=0.99,  # Best fit position: flush right inside plot area
                    y=np.log10(t_val),  # Explicit log conversion for Y placement
                    yref="y",
                    text=f"<b>🎯 {t_col} ({t_val:,.0f})</b>",
                    #text=f"<b>{"Target"}({t_val:,.0f})</b>",
                    #text=f"<b>🎯 {t_col}</b><br>({t_val:,.0f})",
                    showarrow=False,
                    font=dict(color=color, size=11),
                    xanchor="right",
                    yanchor="bottom",
                    bgcolor="rgba(255, 255, 255, 0.8)",  # Soft background so text stays legible over bars
                )
            )

        yaxis_config = get_log_axis_config(agg_df, p_targets)

        return traces, shapes, annotations, yaxis_config

    initial_traces, initial_shapes, initial_annotations, initial_yaxis = (
        build_chart_elements(period)
    )
    fig = go.Figure(data=initial_traces)

    dropdown_buttons = []
    for p_key, p_cfg in periods_config.items():
        p_traces, p_shapes, p_annotations, p_yaxis = build_chart_elements(p_key)

        dropdown_buttons.append(
            dict(
                label=p_cfg["label"],
                method="update",
                args=[
                    {
                        "x": [t.x for t in p_traces],
                        "y": [t.y for t in p_traces],
                        "text": [t.text for t in p_traces],
                    },
                    {
                        # "title.text": f"Achievement vs Target ({p_key})",
                        "title.text": f"<b>Target vs Achievement<b>",
                        "shapes": p_shapes,
                        "annotations": p_annotations,
                        "yaxis": p_yaxis,
                    },
                ],
            )
        )

    fig.update_layout(
        title=dict(
            # text=f"Achievement vs Target ({period})",
            text=f"<b>Target vs Achievement<b>",
            font=dict(size=18),
            x=0.50,
            y=0.95,
        ),
        xaxis_title="Reporting Period",
        yaxis_title="Number of Cases",
        template="plotly_white",
        hovermode="x unified",
        barmode="group",
        bargap=0.2,
        bargroupgap=0.1,
        shapes=initial_shapes,
        annotations=initial_annotations,
        yaxis=initial_yaxis,
        margin=dict(t=80, b=40, l=80, r=80),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0
        ),
        updatemenus=[
            dict(
                type="dropdown",
                active=list(periods_config.keys()).index(period),
                x=0.78,
                xanchor="right",
                y=1.3,
                yanchor="top",
                showactive=True,
                buttons=dropdown_buttons,
            )
        ],
    )

    return fig

# targetVSachievement = plotly_achievement_target_dropdown(
#     dataframe = progress,
#     achievement_columnList = ["Examined Cases Achievement", "Notified Cases Achievement", "BC Cases Achievement"],
#     target_columnList = ["Examined Cases Target","Notified Cases Target","BC Cases Target"],
#     period = "Monthly",
#     date_col = "ReportingDate")
# targetVSachievement.show()

Plotly_Performance_Heatmap

In [42]:
def plotly_variance_heatmap(
    df: pd.DataFrame, 
    indicators: list = ['Examined Cases', 'Notified Cases', 'BC Cases'],
    date_column: str = 'ReportingDate',
    color_scale_range: tuple = None  # Optional tuple e.g., (0, 300). Default is (min_pct, max_pct)
    ):

    data = df.copy()
    data['Tsp'] = data['Team'] + ' - ' + data['Tsp'] + ' - ' + data['TargetCategory']
    data[date_column] = pd.to_datetime(data[date_column])
    
    records = []
    
    for ind in indicators:
        t_col, a_col = f"{ind} Target", f"{ind} Achievement"
        
        if t_col in data.columns and a_col in data.columns:
            # 1. Isolate rows where achievement data is reported (> 0)
            achieve_data = data[data[a_col] > 0]
            
            if achieve_data.empty:
                continue  # Skip indicator if no achievement records exist
                
            # 2. Determine location-specific active date range (Min & Max Achievement Dates)
            date_bounds = achieve_data.groupby(['Tsp', 'Clinic'])[date_column].agg(
                min_date='min', 
                max_date='max'
            ).reset_index()
            
            # 3. Merge date bounds back to slice target/achievement data within active period
            ind_df = data[['Tsp', 'Clinic', date_column, t_col, a_col]].merge(
                date_bounds, on=['Tsp', 'Clinic'], how='inner'
            )
            
            # Filter rows within each location's active reporting window
            filtered_df = ind_df[
                (ind_df[date_column] >= ind_df['min_date']) & 
                (ind_df[date_column] <= ind_df['max_date'])
            ]
            
            # 4. Aggregate within active reporting period
            summary = filtered_df.groupby(['Tsp', 'Clinic', 'min_date', 'max_date'], as_index=False).agg({
                t_col: 'sum',
                a_col: 'sum'
            })
            
            summary['Location'] = summary['Tsp'] + " | " + summary['Clinic']
            summary['Indicator'] = ind
            summary['Target'] = summary[t_col]
            summary['Achievement'] = summary[a_col]
            
            # Calculate Progress %
            summary['Progress_Pct'] = np.where(
                summary['Target'] > 0, 
                (summary['Achievement'] / summary['Target']) * 100, 
                0.0
            )
            
            # Date range display string
            summary['Period'] = (
                summary['min_date'].dt.strftime('%b %d, %Y') + " - " + 
                summary['max_date'].dt.strftime('%b %d, %Y')
            )
            
            records.append(summary[['Location', 'Indicator', 'Target', 'Achievement', 'Progress_Pct', 'Period']])
            
    if not records:
        raise ValueError("No valid achievement data found across the specified indicators.")
        
    combined_df = pd.concat(records, ignore_index=True)
    
    # 5. Pivot matrices for heatmap layout
    pct_matrix = combined_df.pivot(index='Location', columns='Indicator', values='Progress_Pct')
    target_matrix = combined_df.pivot(index='Location', columns='Indicator', values='Target')
    achieve_matrix = combined_df.pivot(index='Location', columns='Indicator', values='Achievement')
    period_matrix = combined_df.pivot(index='Location', columns='Indicator', values='Period')
    
    # Enforce original 'indicators' list ordering on the x-axis (filtering out indicators not in data)
    ordered_cols = [ind for ind in indicators if ind in pct_matrix.columns]
    
    pct_matrix = pct_matrix.reindex(columns=ordered_cols).fillna(0)
    target_matrix = target_matrix.reindex(columns=ordered_cols).fillna(0)
    achieve_matrix = achieve_matrix.reindex(columns=ordered_cols).fillna(0)
    period_matrix = period_matrix.reindex(columns=ordered_cols).fillna("N/A")
    
    # 6. Determine dynamic vs user-defined color scale range (zmin, zmax)
    if color_scale_range is not None and isinstance(color_scale_range, (tuple, list)) and len(color_scale_range) == 2:
        z_min, z_max = color_scale_range
    else:
        z_min = float(pct_matrix.values.min())
        z_max = float(pct_matrix.values.max())
    
    # 7. Build multi-line text labels for cells
    text_matrix = []
    hover_matrix = []
    
    for loc in pct_matrix.index:
        row_text = []
        row_hover = []
        for ind in pct_matrix.columns:
            tgt = int(target_matrix.loc[loc, ind]) if loc in target_matrix.index and ind in target_matrix.columns else 0
            ach = int(achieve_matrix.loc[loc, ind]) if loc in achieve_matrix.index and ind in achieve_matrix.columns else 0
            pct = pct_matrix.loc[loc, ind] if loc in pct_matrix.index and ind in pct_matrix.columns else 0.0
            prd = period_matrix.loc[loc, ind] if loc in period_matrix.index and ind in period_matrix.columns else "N/A"
            
            # Display text inside cell
            cell_str = f"{ach:,} (Targeted {tgt:,}) <br><b>{pct:.1f}%</b>"
            row_text.append(cell_str)
            
            # Tooltip details
            hover_str = (
                f"<b>Location:</b> {loc}<br>"
                f"<b>Indicator:</b> {ind}<br>"
                f"<b>Active Window:</b> {prd}<br>"
                f"<b>Achievement:</b> {ach:,}<br>"
                f"<b>Target:</b> {tgt:,}<br>"
                f"<b>Progress Rate:</b> {pct:.1f}%"
            )
            row_hover.append(hover_str)
            
        text_matrix.append(row_text)
        hover_matrix.append(row_hover)
        
    # Custom Red -> White -> Green Color Scale
    red_white_green = [
        [0.0, '#D9381E'],  # Red (Low values / 0%)
        [0.5, '#FFFFFF'],  # White (Midpoint)
        [1.0, '#2E7D32']   # Green (High values)
    ]
        
    # 8. Render Plotly Heatmap
    fig = go.Figure(data=go.Heatmap(
        z=pct_matrix.values,
        x=list(pct_matrix.columns),
        y=list(pct_matrix.index),
        text=text_matrix,
        texttemplate="%{text}",
        textfont={"size": 11},
        hoverinfo="text",
        hovertext=hover_matrix,
        colorscale=red_white_green,
        zmin=z_min,
        zmax=z_max,
        colorbar=dict(title="% Target Met")
    ))
    
    # Global period bounds string for title
    # overall_min = combined_df['Period'].str.split(' - ').str[0].min()
    # overall_max = combined_df['Period'].str.split(' - ').str[1].max()
    overall_min = pd.to_datetime(combined_df['Period'].str.split(' - ').str[0]).min().strftime('%b-%Y')
    overall_max = pd.to_datetime(combined_df['Period'].str.split(' - ').str[1]).max().strftime('%b-%Y')
    
    fig.update_layout(
    title=dict(
        text=f"<b>Performance Heatmap (From: {overall_min} to {overall_max})</b>",  # Added 'text=' and fixed closing </b> tag
        font=dict(size=18),
        x=0.50,
        y=0.93,
        xanchor="center"  # Optional: ensures x=0.50 perfectly centers the title
    ),
    template="plotly_white",
    xaxis_title="Indicators",
    yaxis_title="Tsp | Clinic",
    xaxis=dict(categoryorder='array', categoryarray=ordered_cols),
    height=max(450, len(pct_matrix) * 50),
    margin=dict(l=150, r=40, t=90, b=40)
    )
    
    return fig


# fig = plotly_variance_heatmap(progress).show()


Plotly_TargetAchievement_AllChart

In [43]:

def plotly_target_achievement_allcharts(
    dataframe,
    date_config,
    bar_configs,
    optional_percentage=True,
    percentage_calc=None,
    freq="Month"
):
    df = dataframe.copy()

    date_col = list(date_config)[0]
    date_label = date_config[date_col]

    df[date_col] = pd.to_datetime(
        df[date_col],
        errors="coerce"
    )

    f = freq.lower()

    if f in ["month", "m"]:
        period = df[date_col].dt.to_period("M")
        label_format = "%Y-%b"

    elif f in ["quarter", "q"]:
        period = df[date_col].dt.to_period("Q")
        label_format = "%Y-Q%q"

    elif f in ["semi-annual", "semi_annual", "sa"]:
        period = (
            df[date_col].dt.year.astype(str)
            + "-"
            + df[date_col].dt.month.map(
                lambda x: "S1" if x <= 6 else "S2"
            )
        )
        label_format = None

    elif f in ["annual", "year", "a", "y"]:
        period = df[date_col].dt.to_period("Y")
        label_format = "%Y"

    else:
        raise ValueError(
            "freq must be Month, Quarter, Semi-Annual or Annual"
        )

    df_grouped = (
        df.groupby(period)
        .sum(numeric_only=True)
        .reset_index()
    )

    if label_format:
        df_grouped[date_col] = (
            df_grouped[date_col]
            .dt.strftime(label_format)
        )
    else:
        df_grouped.rename(
            columns={df_grouped.columns[0]: date_col},
            inplace=True
        )

    charts = {}

    for config in bar_configs:

        target_col = next(
            (
                col for col, label in config.items()
                if "target" in col.lower()
                or "target" in label.lower()
            ),
            None
        )

        achievement_col = next(
            (
                col for col in config
                if col != target_col
            ),
            None
        )

        if not target_col or not achievement_col:
            raise ValueError(
                f"Could not identify Target/Achievement "
                f"columns: {config}"
            )

        indicator = (
            achievement_col
            .replace(" Achievement", "")
            .replace("_Achievement", "")
            .replace("Achievement", "")
            .strip()
        )

        target = pd.to_numeric(
            df_grouped[target_col],
            errors="coerce"
        ).fillna(0)

        achievement = pd.to_numeric(
            df_grouped[achievement_col],
            errors="coerce"
        ).fillna(0)

        target_total = target.sum()
        achievement_total = achievement.sum()

        progress_total = (
            achievement_total / target_total * 100
            if target_total > 0
            else None
        )

        progress_label = (
            f"{progress_total:.0f}%"
            if progress_total is not None
            else "N/A"
        )

        fig = go.Figure()

        fig.add_trace(
            go.Bar(
                x=df_grouped[date_col].astype(str),
                y=achievement,
                name=f"Achievement ({achievement_total:,.0f})",
                text=achievement.map(
                    lambda x: f"{x:,.0f}"
                ),
                textposition="inside",
                insidetextanchor="middle"
            )
        )

        fig.add_trace(
            go.Scatter(
                x=df_grouped[date_col].astype(str),
                y=target,
                name=f"Target ({target_total:,.0f})",
                mode="lines+markers+text",
                text=target.map(
                    lambda x: f"{x:,.0f}"
                ),
                textposition="top center",
                line=dict(width=2),
                marker=dict(size=7)
            )
        )

        if (
            optional_percentage
            and percentage_calc
            and indicator in percentage_calc
        ):

            num_col, den_col = percentage_calc[indicator]

            if (
                num_col in df_grouped.columns
                and den_col in df_grouped.columns
            ):

                num = pd.to_numeric(
                    df_grouped[num_col],
                    errors="coerce"
                )

                den = pd.to_numeric(
                    df_grouped[den_col],
                    errors="coerce"
                )

                pct = pd.Series(
                    pd.NA,
                    index=df_grouped.index,
                    dtype="Float64"
                )

                valid = (
                    num.notna()
                    & den.notna()
                    & (num > 0)
                    & (den > 0)
                )

                pct.loc[valid] = (
                    num.loc[valid]
                    / den.loc[valid]
                    * 100
                ).round(0)

                fig.add_trace(
                    go.Scatter(
                        x=df_grouped[date_col].astype(str),
                        y=pct,
                        name=f"Progress ({progress_label})",
                        mode="lines+markers+text",
                        text=pct.map(
                            lambda x:
                            f"{x:.0f}%"
                            if pd.notna(x)
                            else ""
                        ),
                        textposition="top center",
                        line=dict(
                            dash="dash",
                            width=2
                        ),
                        marker=dict(size=7),
                        connectgaps=False,
                        yaxis="y2"
                    )
                )

        fig.update_layout(
            title=dict(
                text=f'<b>{indicator}</b>',
                x=0.5,
                xanchor="center"
            ),
            height=450,
            barmode="group",
            hovermode="x unified",
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="center",
                x=0.5
            ),
            template="plotly_white",
            margin=dict(
                t=100,
                b=70,
                l=60,
                r=60
            )
        )

        fig.update_xaxes(
            title_text=date_label
        )

        fig.update_yaxes(
            title_text="# of Cases",
            rangemode="tozero"
        )

        if optional_percentage:
            fig.update_layout(
                yaxis2=dict(
                    title="Progress %",
                    overlaying="y",
                    side="right",
                    rangemode="tozero",
                    ticksuffix="%"
                )
            )

        charts[indicator] = fig

    return charts

# charts = plotly_target_achievement_allcharts(
#     dataframe=progress,
#     date_config={
#         "ReportingDate": "Reporting Period"
#     },
#     bar_configs=[
#         {
#             "Examined Cases Target": "Examined Cases Target",
#             "Examined Cases Achievement": "Examined Cases Achievement"
#         },
#         {
#             "Notified Cases Target": "Notified Cases Target",
#             "Notified Cases Achievement": "Notified Cases Achievement"
#         },
#         {
#             "BC Cases Target": "BC Cases Target",
#             "BC Cases Achievement": "BC Cases Achievement"
#         }
#     ],
#     optional_percentage=True,
#     percentage_calc={
#         "Examined Cases": (
#             "Examined Cases Achievement",
#             "Examined Cases Target"
#         ),
#         "Notified Cases": (
#             "Notified Cases Achievement",
#             "Notified Cases Target"
#         ),
#         "BC Cases": (
#             "BC Cases Achievement",
#             "BC Cases Target"
#         )
#     },
#     freq="Month"
# )

# charts["Examined Cases"].show()
# charts["Notified Cases"].show()
# charts["BC Cases"].show()

Plotly_Distribution_AgeSex

In [44]:
def plotly_gender_agegroup(
    dataframe,
    sex,
    age,
    xaxis_interval=200
):
    df = dataframe.copy()

    if xaxis_interval <= 0:
        raise ValueError("xaxis_interval must be greater than 0")

    df[age] = pd.to_numeric(
        df[age],
        errors="coerce"
    )

    bins = [
        -1, 4, 9, 14, 24, 34, 44, 54, 64, float("inf")
    ]

    labels = [
        "0-4",
        "5-9",
        "10-14",
        "15-24",
        "25-34",
        "35-44",
        "45-54",
        "55-64",
        "≥ 65"
    ]

    df["AgeGroup"] = pd.cut(
        df[age],
        bins=bins,
        labels=labels
    )

    df[sex] = (
        df[sex]
        .astype(str)
        .str.upper()
        .str.strip()
        .replace({
            "MALE": "M",
            "FEMALE": "F"
        })
    )

    tab = pd.crosstab(
        df["AgeGroup"],
        df[sex]
    ).reindex(
        labels,
        fill_value=0
    )

    male = -tab.get(
        "M",
        pd.Series(0, index=labels)
    )

    female = tab.get(
        "F",
        pd.Series(0, index=labels)
    )

    male_total = abs(male).sum()
    female_total = female.sum()

    ratio = (
        male_total / female_total
        if female_total > 0
        else 0
    )

    max_value = max(
        abs(male).max(),
        female.max()
    )

    axis_max = (
        int((max_value + xaxis_interval - 1)
        // xaxis_interval)
        * xaxis_interval
    )

    tickvals = list(
        range(
            -axis_max,
            axis_max + xaxis_interval,
            xaxis_interval
        )
    )

    ticktext = [
        f"{abs(x):,}"
        for x in tickvals
    ]

    fig = go.Figure()

    fig.add_bar(
        y=labels,
        x=male,
        orientation="h",
        name=f"Male ({male_total:,})",
        text=abs(male),
        textposition="outside",
        cliponaxis=False
    )

    fig.add_bar(
        y=labels,
        x=female,
        orientation="h",
        name=f"Female ({female_total:,})",
        text=female,
        textposition="outside",
        cliponaxis=False
    )

    fig.update_layout(
        title=dict(
            text = f"<b>Disaggregation by Sex and Age Group</b>",
            font=dict(size=18),
            x=0.50,
            y=0.95,
            xanchor="center"
        ),
        barmode="relative",
        template="plotly_white",
        xaxis=dict(
            title=f"Ratio - Male ({ratio:.2f} : 1) Female",
            range=[
                -axis_max * 1.10,
                axis_max * 1.10
            ],
            tickmode="array",
            tickvals=tickvals,
            ticktext=ticktext,
            zeroline=True,
            zerolinewidth=2,
            showgrid=True
        ),

        yaxis=dict(
            title="Age Group",

            categoryorder="array",

            categoryarray=labels
        ),

        legend=dict(
            orientation="h",
            y=1.2,
            x=0.5,
            xanchor="center"
        ),

        margin=dict(
            l=70,
            r=70,
            t=100,
            b=70
        )
    )

    return fig

# fig = plotly_gender_agegroup(
#     df_filtered,
#     "Sex",
#     "Age",
#     500
# )

# fig.show()

Plotly_Distribution_H-Bar

In [45]:
def plotly_stack_bar(
    dataframe,
    columns,
    rename_dict=None,  # <-- 1. ADDED OPTIONAL PARAMETER
    exclude_blank=True,
    orientation="v",
    title="Disaggregation by Category"
):
    df = dataframe.copy()

    # <-- 2. APPLY RENAMING EARLY & UPDATE COLUMNS LIST
    if rename_dict:
        df = df.rename(columns=rename_dict)
        if isinstance(columns, str):
            columns = [rename_dict.get(columns, columns)]
        else:
            columns = [rename_dict.get(col, col) for col in columns]
    elif isinstance(columns, str):
        columns = [columns]

    missing = [
        col for col in columns
        if col not in df.columns
    ]

    if missing:
        raise KeyError(
            f"Missing columns: {missing}"
        )

    orientation = orientation.lower()

    if orientation not in ["v", "h"]:
        raise ValueError(
            "orientation must be 'v' or 'h'"
        )

    fig = go.Figure()

    for col in columns:

        data = (
            df[col]
            .astype("string")
            .str.strip()
        )

        if exclude_blank:
            data = data[
                data.notna()
                & data.ne("")
            ]
        else:
            data = data.fillna("Blank")
            data = data.replace("", "Blank")

        if data.empty:
            continue

        counts = (
            data.value_counts()
            .sort_values(ascending=False)
        )

        total = counts.sum()

        percentages = counts / total * 100

        for category in counts.index:

            count = counts[category]
            percent = percentages[category]

            label = (
                f"{category}"
                f"<br>{count:,}"
                f"<br>{percent:.1f}%"
            )

            hover = (
                f"<b>{col}</b>"
                f"<br>Category: {category}"
                f"<br>Total: {count:,}"
                f"<br>Percent: {percent:.1f}%"
                f"<br>Column Total: {total:,}"
                "<extra></extra>"
            )

            text_angle = 0 if percent >= 12 else -90

            if orientation == "v":

                fig.add_trace(
                    go.Bar(
                        x=[col],
                        y=[percent],
                        name=str(category),
                        text=[label],
                        textposition="inside",
                        insidetextanchor="middle",
                        textangle=text_angle,
                        hovertemplate=hover,
                        showlegend=False
                    )
                )

            else:

                fig.add_trace(
                    go.Bar(
                        y=[col],
                        x=[percent],
                        orientation="h",
                        name=str(category),
                        text=[label],
                        textposition="inside",
                        insidetextanchor="middle",
                        textangle=text_angle,
                        hovertemplate=hover,
                        showlegend=False
                    )
                )

    fig.update_layout(
        title=dict(
            text=f'<b>{title}</b>',
            font=dict(size=18),
            x=0.5,
            y=0.95,
            xanchor="center"
        ),

        barmode="stack",
        template="plotly_white",
        hovermode="closest",
        showlegend=False,

        margin=dict(
            l=50,
            r=50,
            t=80,
            b=50
        )
    )

    if orientation == "v":

        fig.update_yaxes(
            title=None,
            range=[0, 100],
            showticklabels=False,
        )

        fig.update_xaxes(
            title=None,
            categoryorder="array",
            categoryarray=columns
        )

    else:

        fig.update_xaxes(
            title=None,
            range=[0, 100],
            showticklabels=False,
        )

        fig.update_yaxes(
            title=None,
            categoryorder="array",
            categoryarray=columns
        )

    return fig


Plotly_Sankey

In [46]:

def function_sankey_cascade_log(dataframe,criteria_dict,title="TB Cascade of Care",width=1000,height=500,log_base=10):

    df = dataframe.copy()
    if ("Reasonforexamination" in df.columns and "Referralfor" in df.columns):
        mask = ((df["Reasonforexamination"] == "Diagnosis") & (df["Referralfor"].isna() | (df["Referralfor"].astype(str).str.strip()=="")))
        df.loc[mask,"Referralfor"] = "Presumptive"

    stages = list(criteria_dict.keys())

    node_map = {}

    labels = []
    node_counts = []
    node_colors = []

    colors = ["#4C78A8","#F58518","#54A24B","#E45756","#72B7B2","#B279A2"]
    node_id = 0
    for i, stage in enumerate(stages):
        valid_values = criteria_dict[stage]
        for value in valid_values:
            count = (df[stage].astype(str).eq(str(value)).sum())
            node_map[(stage,value)] = node_id
            labels.append(
                f"<b>{value}</b><br>"
                f"{count:,} "
                f"({count/len(df)*100:.1f}%)"
            )
            node_counts.append(count)
            node_colors.append(colors[i % len(colors)])
            node_id += 1

    source=[]
    target=[]
    values=[]
    original=[]
    percentages=[]

    for i in range(len(stages)-1):
        stage1 = stages[i]
        stage2 = stages[i+1]
        temp = (
            df[df[stage1].isin(criteria_dict[stage1]) & df[stage2].isin(criteria_dict[stage2])]
            .groupby([stage1,stage2])
            .size()
            .reset_index(name="Count")
        )
        for _, row in temp.iterrows():
            count = row["Count"]
            source_count = (df[df[stage1].eq(row[stage1])].shape[0])
            retention = (
                count/source_count*100
                if source_count>0
                else 0
            )
            source.append(node_map[(stage1,row[stage1])])
            target.append(node_map[(stage2,row[stage2])])
            values.append(np.log(count+1)/np.log(log_base))
            original.append(count)
            percentages.append(retention)

    fig = go.Figure(
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=15,              # Reduced pad to allow taller nodes
                    thickness=60,        # <-- INCREASED: Expands node box width to fit labels inside
                    align="center",      # <-- ADDED: Forces text inside node boundaries
                    label=labels,
                    color=node_colors,
                    line=dict(color="black", width=1)
                ),
                link=dict(
                    source=source,
                    target=target,
                    value=values,
                    customdata=np.column_stack((original, percentages)),
                    hovertemplate=(
                        "<b>%{source.label}</b>"
                        "<br>↓<br>"
                        "<b>%{target.label}</b>"
                        "<br><br>"
                        "Patients: <b>%{customdata[0]:,}</b>"
                        "<br>"
                        "Retention: <b>%{customdata[1]:.1f}%</b>"
                        "<extra></extra>"
                    )
                )
            )
        )




    fig.update_layout(title=dict(text=f'<b>{title}</b>',
                                 font=dict(size=18),
                                 x=0.5,
                                 y=0.95,
                                 xanchor="center"),
                      width=width,
                      height=height,
                      template="plotly_white",
                      dragmode="zoom",
                      margin=dict(l=30,r=30,t=70,b=30))
    
    return fig

# colSankey = {"VOL":["Volunteer Referral","Walk-In"],"Referralfor":["CI","Presumptive"],"Case":["TB"],"Bact_status":["BC","CD"],"Treatmentreferral":["Registered"]}
# df_sankey = df_filtered[df_filtered["Reasonforexamination"]=="Diagnosis"]
# fig = function_sankey_cascade_log(
#     dataframe=df_sankey,
#     criteria_dict=colSankey,
#     title="TB Cascade: Diagnosis to Treatment Registration",
#     log_base=10
# )
# fig.show(
#     config={
#         "scrollZoom": True,
#         "displaylogo": False
#     }
# )

Plotly_HeatMap

In [47]:

def function_heatmap(dataframe: pd.DataFrame,Xaxis: str,Yaxis: str,exclude_blank: bool = True,colorscale: str = "Blues",) -> go.Figure:

    df = dataframe.copy()

    # Normalize empty strings (" ", "") to NaN for string/object columns
    for col in [Xaxis, Yaxis]:
        if (
            df[col].dtype == "object"
            or isinstance(df[col].dtype, pd.CategoricalDtype)
            or pd.api.types.is_string_dtype(df[col])
        ):
            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .replace(r"^\s*$", np.nan, regex=True)
            )

    if exclude_blank:
        # Drop rows ONLY if BOTH columns are missing/blank
        df = df.dropna(subset=[Xaxis, Yaxis], how="all")

    # Fill missing values with "Not Done"
    df[Xaxis] = df[Xaxis].fillna("Not Done")
    df[Yaxis] = df[Yaxis].fillna("Not Done")

    # 1. Compute Crosstab including margins (Totals)
    counts = pd.crosstab(
        df[Yaxis],
        df[Xaxis],
        dropna=False,
        margins=True,
        margins_name="Total",
    )

    row_order = [idx for idx in counts.index if str(idx) not in ["Total", "Not Done"]]
    if "Not Done" in counts.index:
        row_order.insert(0, "Not Done")
    row_order.insert(0, "Total")
    counts = counts.loc[row_order]

    # Column order (X-axis): Regular values first, then Not Done, then Total at the end
    col_order = [col for col in counts.columns if str(col) not in ["Total", "Not Done"]]
    if "Not Done" in counts.columns:
        col_order.append("Not Done")
    col_order.append("Total")
    counts = counts[col_order]

    grand_total = counts.loc["Total", "Total"]

    # 3. Build text, hover matrices, and log-transformed z-matrix
    text_matrix = []
    hover_matrix = []

    # Apply log10 scaling to data cells (log1p handles 0 counts safely)
    z_values = np.log10(counts.values.astype(float) + 1)

    for i, row_label in enumerate(counts.index):
        text_row = []
        hover_row = []
        for j, col_label in enumerate(counts.columns):
            c_val = counts.iloc[i, j]

            is_total_row = str(row_label) == "Total"
            is_total_col = str(col_label) == "Total"

            # Set total cells to NaN so they stay uncolored
            if is_total_row or is_total_col:
                z_values[i, j] = np.nan

            # Positional column total lookup (row 0 is 'Total')
            col_total = counts.iloc[0, j]

            if is_total_row and is_total_col:
                cell_text = f"<b>{c_val}</b><br>(100.0%)"
                hover_text = f"<b>Grand Total</b>: {c_val}"
            elif is_total_row:
                pct = (c_val / grand_total * 100) if grand_total > 0 else 0
                cell_text = f"<b>{c_val}</b><br>({pct:.1f}%)"
                hover_text = (
                    f"<b>Column Total ({col_label})</b>: {c_val} ({pct:.1f}% of total)"
                )
            elif is_total_col:
                pct = (c_val / grand_total * 100) if grand_total > 0 else 0
                cell_text = f"<b>{c_val}</b><br>({pct:.1f}%)"
                hover_text = (
                    f"<b>Row Total ({row_label})</b>: {c_val} ({pct:.1f}% of total)"
                )
            else:
                pct = (c_val / col_total * 100) if col_total > 0 else 0
                cell_text = f"<b>{c_val}</b><br>({pct:.1f}%)"
                hover_text = (
                    f"<b>{Yaxis}</b>: {row_label}<br>"
                    f"<b>{Xaxis}</b>: {col_label}<br>"
                    f"<b>Count</b>: {c_val}<br>"
                    f"<b>Col %</b>: {pct:.1f}%"
                )

            text_row.append(cell_text)
            hover_row.append(hover_text)

        text_matrix.append(text_row)
        hover_matrix.append(hover_row)

    # 4. Build Plotly Heatmap
    fig = go.Figure(
        data=go.Heatmap(
            z=z_values,
            x=[str(col) for col in counts.columns],
            y=[str(idx) for idx in counts.index],
            text=text_matrix,
            texttemplate="%{text}",
            hoverinfo="text",
            hovertext=hover_matrix,
            colorscale=colorscale,
            showscale=False,
        )
    )

    # 5. Layout configuration
    fig.update_layout(title=dict(text=f"<b>Gene Xpert Result on Chest X-ray Findings</b>",
                   font=dict(size=18),
                   x=0.5,
                   y=0.95,
                   xanchor="center"),
        xaxis_title=f"{Xaxis}",
        yaxis_title=f"{Yaxis}",
        template="plotly_white",
        xaxis=dict(side="bottom"),
        yaxis=dict(autorange="reversed"),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    return fig
# function_heatmap(df_filtered,"CXRresult","GeneXpertresult")

Plotly_Table

In [48]:

def plotly_table_count_percent(
    df: pd.DataFrame, 
    column_list: list, 
    optional_exclude_blank: bool = True,
    optional_include_total: bool = True
):
    """
    Renders a Plotly Table showing Column Name, Category, Count, and Percent 
    for multiple columns, with the Total row positioned at the top of each column group.
    
    Parameters:
    - df: pandas DataFrame
    - column_list: list of column names to aggregate
    - optional_exclude_blank: bool, filters out NaN and empty string values (default True)
    - optional_include_total: bool, adds a Total row at the top of each column group (default True)
    """
    table_rows = []
    
    for col in column_list:
        data = df[col].copy()
        
        # Handle blank/null values
        if optional_exclude_blank:
            data = data.dropna()
            if data.dtype == 'object' or isinstance(data.dtype, pd.CategoricalDtype):
                data = data[~data.astype(str).str.strip().isin(['', 'None', 'nan', 'NaN'])]
        
        # Compute value counts and percentages
        counts = data.value_counts(dropna=not optional_exclude_blank).reset_index()
        counts.columns = ['Category', 'Count']
        total_count = counts['Count'].sum()
        counts['Percent'] = (counts['Count'] / total_count * 100) if total_count > 0 else 0.0
        
        # 1. Add Total row FIRST if requested
        if optional_include_total:
            table_rows.append({
                'Column Name': col,  # Place column name on Total row
                'Category': "Total",
                'Count': total_count,
                'Percent': 100.0 if total_count > 0 else 0.0,
                'Is_Total': True
            })
            
        # 2. Add individual Category rows
        for i, row in counts.iterrows():
            # If Total row exists, blank out column name; otherwise show it on the first category row
            col_display = col if (i == 0 and not optional_include_total) else ""
            
            table_rows.append({
                'Column Name': col_display,
                'Category': str(row['Category']),
                'Count': row['Count'],
                'Percent': row['Percent'],
                'Is_Total': False
            })

    result_df = pd.DataFrame(table_rows)
    
    # Formatted cell content
    formatted_col_name = [f"<b>{r['Column Name']}</b>" for _, r in result_df.iterrows()]
    formatted_category = [f"<b>{r['Category']}</b>" if r['Is_Total'] else r['Category'] for _, r in result_df.iterrows()]
    formatted_counts = [f"<b>{r['Count']:,}</b>" if r['Is_Total'] else f"{r['Count']:,}" for _, r in result_df.iterrows()]
    formatted_percent = [f"<b>{r['Percent']:.1f}%</b>" if r['Is_Total'] else f"{r['Percent']:.1f}%" for _, r in result_df.iterrows()]
    
    # Background color formatting
    fill_colors = []
    for _, r in result_df.iterrows():
        if r['Is_Total']:
            fill_colors.append('#E1EBF5')  # Light blue/gray highlight for Total header row
        else:
            fill_colors.append('#FFFFFF')  # White background for category rows
            
    # Create Plotly Table
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=["<b>Column Name</b>", "<b>Category</b>", "<b>Count</b>", "<b>Percent</b>"],
            fill_color='#1F77B4',
            font=dict(color='white', size=13),
            align=['left', 'left', 'right', 'right']
        ),
        cells=dict(
            values=[formatted_col_name, formatted_category, formatted_counts, formatted_percent],
            fill_color=[fill_colors] * 4,
            font=dict(color='black', size=12),
            align=['left', 'left', 'right', 'right'],
            height=26
        )
    )])
    
    fig.update_layout(
        title="Summary Table: Count and Percent Breakdown",
        margin=dict(l=20, r=20, t=50, b=20)
    )
    return fig

# tbl_pc = plotly_table_count_percent(
#     df = df_filtered, 
#     column_list = ['Sex','CXRresult','GeneXpertresult','Case','Placeforreferral'], 
#     optional_exclude_blank = True,
#     optional_include_total = True
# )
# tbl_pc.show()

Plotly_Funnel

In [49]:

def plotly_funnel(
    df: pd.DataFrame, 
    funnel_column_criteria: dict, 
    rename_column: list, 
    column_funnel: str
):

    raw_step_names = list(funnel_column_criteria.keys())
    
    # Validation check for rename_column list length
    if len(rename_column) != len(raw_step_names):
        raise ValueError(
            f"Length of `rename_column` ({len(rename_column)}) must match "
            f"the number of stages in `funnel_column_criteria` ({len(raw_step_names)})."
        )
    
    # 1. Get unique segments from column_funnel
    segments = df[column_funnel].dropna().unique().tolist()
    num_segments = len(segments)
    
    if num_segments == 0:
        raise ValueError(f"No unique values found in group column: '{column_funnel}'")
    
    # 2. Setup subplot grid (1 row, N columns)
    fig = make_subplots(
        rows=1, 
        cols=num_segments,
        # subplot_titles=[f"<b>{column_funnel}: {seg}</b>" for seg in segments],
        subplot_titles=[f"<b>{seg}</b>" for seg in segments],
        shared_yaxes=True
    )
    
    # 3. Iterate through segments and build individual funnels
    for i, seg in enumerate(segments, start=1):
        seg_df = df[df[column_funnel] == seg]
        raw_counts = []
        
        # Calculate raw counts per funnel stage
        for col, criteria in funnel_column_criteria.items():
            criteria_list = criteria if isinstance(criteria, list) else [criteria]
            count = seg_df[col].isin(criteria_list).sum()
            raw_counts.append(count)
            
        # Log10 transformation for bar widths (log10(count + 1) prevents log(0))
        log_counts = np.log10(np.array(raw_counts) + 1).tolist()
        
        # Calculate percentages based on raw counts
        initial_count = raw_counts[0] if len(raw_counts) > 0 and raw_counts[0] > 0 else 1
        pct_initial = [(cnt / initial_count) * 100 for cnt in raw_counts]
        
        # Format display text and hover text using custom labels from rename_column
        display_text = [
            f"{cnt:,} ({pct:.1f}%)" for cnt, pct in zip(raw_counts, pct_initial)
        ]
        hover_text = [
            f"<b>Stage:</b> {label}<br><b>Raw Count:</b> {cnt:,}<br><b>% of Initial:</b> {pct:.1f}%"
            for label, cnt, pct in zip(rename_column, raw_counts, pct_initial)
        ]
        
        # Add funnel trace to subplot using renamed stages for y-axis
        fig.add_trace(
            go.Funnel(
                name=str(seg),
                y=rename_column,        # Custom labels shown on y-axis
                x=log_counts,           # Log values control bar widths
                text=display_text,       # Actual counts & percentages on bars
                textinfo="text",
                hoverinfo="text",
                hovertext=hover_text
            ),
            row=1, 
            col=i
        )
        
    # 4. Layout configuration
    fig.update_layout(
        title=dict(text=f"<b>Cascade of Care Analysis by {column_funnel}</b>",
                        font=dict(size=18),
                        x=0.5,
                        y=0.95,
                        xanchor='center'),
        showlegend=False,
        template="plotly_white",
        margin=dict(l=40, r=40, t=80, b=40)
    )
    
    # Hide log tick values along the bottom x-axes
    fig.update_xaxes(showticklabels=False, title_text="")
    
    return fig

# funnel_column_criteria =  {'Reasonforexamination':['Diagnosis'],
#                            'Cxrr':['Requested'],
#                            'CXRresult':['TB Suspect','TB Healed','TB Active'],
#                            'Genexpertrequested':['Requested'],
#                            'GeneXpertresult':['N', 'T', 'TT', 'TI', 'RR',],
#                            'Bact_status':['BC'],
#                            'Case':['TB'],
#                            'Treatmentreferral':['Registered']
#                            }
# funnel_column_rename = ['Screening','CXR Request','CXR Abnormality','Gene Request','Gene Result','Bact Confirmed','Notified TB','Treatment Registered']
# fig_funnel = plotly_funnel(df_filtered,funnel_column_criteria,funnel_column_rename,'Symptom')
# fig_funnel.show()

Plotly Funnel OPD

In [50]:

def plot_clinic_sankey(df: pd.DataFrame) -> go.Figure:

    data = df.copy()

    # 1. Categorize Attendant Type
    def get_attendant_type(val):
        if pd.isna(val):
            return "Unspecified Attendant"
        val_str = str(val).strip().lower()
        if val_str in ["new","yes", "1", "1.0"]:
            return "New Attendant"
        elif val_str in ["old","no", "2", "2.0"]:
            return "Old Attendant"
        return "Unspecified Attendant"

    data["Attendant_Category"] = data["TypeofPatient1"].apply(get_attendant_type)

    # 2. Categorize Consultation Types
    valid_ht_dm = {"1", "3", "yes", "1.0", "3.0"}
    valid_binary = {"1", "yes", "1.0"}

    def parse_flag(val, valid_set):
        if pd.isna(val):
            return False
        return str(val).strip().lower() in valid_set

    def get_consultation_categories(row):
        categories = []
        is_ht = parse_flag(row.get("HT1"), valid_ht_dm)
        is_dm = parse_flag(row.get("DM1"), valid_ht_dm)
        is_avi = parse_flag(row.get("RTIAVI1"), valid_binary)
        is_general_weakness = parse_flag(row.get("Generalweakness1"), valid_binary)
        is_others = parse_flag(row.get("Other1"), valid_binary)

        if is_ht and is_dm:
            categories.append("HT+DM")
        elif is_ht:
            categories.append("Hypertension")
        elif is_dm:
            categories.append("Diabetes")

        if is_avi:
            categories.append("AVI")
        if is_general_weakness:
            categories.append("General Weakness")
        if is_others:
            categories.append("Others")

        return categories if categories else ["Unspecified/Blank"]

    data["Consultation_List"] = data.apply(
        get_consultation_categories, axis=1
    )
    exploded = data.explode("Consultation_List")

    # 3. Aggregate Link Flows (Source -> Target)
    flow_counts = (
        exploded.groupby(["Attendant_Category", "Consultation_List"])
        .size()
        .reset_index(name="Patient_Count")
    )

    # 4. Map Nodes to Index Positions
    sources = flow_counts["Attendant_Category"].tolist()
    targets = flow_counts["Consultation_List"].tolist()
    unique_nodes = list(dict.fromkeys(sources + targets))

    node_indices = {name: i for i, name in enumerate(unique_nodes)}

    source_idx = [node_indices[s] for s in sources]
    target_idx = [node_indices[t] for t in targets]
    values = flow_counts["Patient_Count"].tolist()

    # 5. Define Custom Node Colors
    color_palette = {
        "New Attendant": "#2b5c8f",
        "Old Attendant": "#d95f02",
        "Unspecified Attendant": "#8c8c8c",
        "Hypertension": "#1f77b4",
        "Diabetes": "#ff7f0e",
        "HT+DM": "#d62728",
        "AVI": "#9467bd",
        "Others": "#2ca02c",
        "Unspecified/Blank": "#7f7f7f",
    }
    node_colors = [
        color_palette.get(node, "#333333") for node in unique_nodes
    ]

    # 6. Construct Sankey Figure
    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    pad=20,
                    thickness=20,
                    line=dict(color="black", width=0.5),
                    label=unique_nodes,
                    color=node_colors,
                ),
                link=dict(
                    source=source_idx,
                    target=target_idx,
                    value=values,
                    color="rgba(180, 180, 180, 0.3)",
                ),
            )
        ]
    )

    fig.update_layout(
        title_text="<b>Patient Flow: Clinic Attendant Type → Consultation Category</b>",
        font_size=12,
        template="plotly_white",
        margin=dict(t=60, b=40, l=40, r=40),
    )

    return fig

Plotyl Bubble

In [51]:

def plot_phc_category_bubble(df, category_col='PHC Category', sep=', ', base_size=8, log_factor=10):
    """
    Generates an interactive Plotly bubble chart showing category co-occurrence.
    Uses log-scaled bubble sizes and displays total category counts along the top X-axis.
    
    Returns:
        plotly.graph_objects.Figure
    """
    # 1. Unpack categories into binary dummy columns & compute co-occurrence matrix
    dummies = df[category_col].dropna().str.get_dummies(sep=sep)
    categories = sorted(dummies.columns)
    dummies = dummies[categories]
    
    co_matrix = dummies.T.dot(dummies)
    
    # 2. Reshape co-occurrence matrix to long format
    co_df = co_matrix.reset_index().melt(id_vars='index')
    co_df.columns = ['Cat_X', 'Cat_Y', 'Count']
    co_df = co_df[co_df['Count'] > 0].copy()
    
    # 3. Apply logarithmic scaling to size smaller bubbles (np.log1p prevents log(0) errors)
    co_df['log_size'] = base_size + (np.log1p(co_df['Count']) * log_factor)
    
    # 4. Extract total count per category for the top X-axis
    totals = [co_matrix.loc[cat, cat] for cat in categories]
    top_totals_text = [str(t) for t in totals]
    
    # 5. Create Scatter Plot
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=co_df['Cat_X'],
        y=co_df['Cat_Y'],
        mode='markers+text',
        text=co_df['Count'].astype(str),
        textposition='middle center',
        textfont=dict(color='black', size=8, family='Arial Bold'),
        marker=dict(
            size=co_df['log_size'],
            color=co_df['Count'],
            colorscale='YlGnBu',
            showscale=True,
            colorbar=dict(title='Intersection Count'),
            line=dict(width=1, color='DarkBlue')
        ),
        hovertemplate="<b>X Category:</b> %{x}<br>" +
                      "<b>Y Category:</b> %{y}<br>" +
                      "<b>Intersection Count:</b> %{text}<extra></extra>"
    ))
    
    # 6. Configure Layout with Secondary Top X-Axis for Total Counts
    fig.update_layout(
        title=dict(text='PHC Category Intersection Matrix', x=0.5, y=0.98, font=dict(size=16)),
        xaxis=dict(
            title='PHC Category',
            tickangle=-45,
            categoryorder='array',
            categoryarray=categories
        ),
        xaxis2=dict(
            title=dict(text='Total Category Count', font=dict(size=12, color='black')),
            overlaying='x',
            side='top',
            tickmode='array',
            tickvals=categories,
            ticktext=top_totals_text,
            tickangle=0,
            tickfont=dict(size=11, color='black', family='Arial Bold')
        ),
        yaxis=dict(
            title='PHC Category',
            categoryorder='array',
            categoryarray=categories
        ),
        width=850,
        height=750,
        template='plotly_white',
        margin=dict(t=120, b=80, l=90, r=80)
    )
    
    return fig

Plotly_Waterfall

In [52]:

def plotly_waterfall(
    df: pd.DataFrame,
    start_dict: dict,
    subtract_dict1: dict,
    add_dict: dict,
    subtract_dict2: dict,
    chart_title: str = "Waterfall Analysis",
):
    """Generates a sequential Plotly Waterfall chart without subtotals,

    using valid Plotly waterfall text positions ('inside' vs 'outside').
    """

    def apply_filter(data, criteria_dict):
        mask = pd.Series(True, index=data.index)
        for col, values in criteria_dict.items():
            if col in data.columns:
                val_list = values if isinstance(values, list) else [values]
                mask &= data[col].isin(val_list)
        return mask

    x_labels = []
    y_values = []
    measure_types = []
    text_labels = []
    raw_counts = []

    running_total = 0

    # Helper function to append steps
    def process_dict(dict_obj, is_subtraction=False):
        nonlocal running_total
        for step_label, criteria in dict_obj.items():
            count = apply_filter(df, criteria).sum()
            if is_subtraction:
                running_total -= count
                val = -count
                txt = f"-{count:,}"
            else:
                running_total += count
                val = count
                txt = f"+{count:,}"

            x_labels.append(step_label)
            y_values.append(val)
            measure_types.append("relative")
            text_labels.append(txt)
            raw_counts.append(count)

    # 1. Process Start Base
    process_dict(start_dict, is_subtraction=False)

    # 2. Process First Subtraction (subtract_dict1)
    process_dict(subtract_dict1, is_subtraction=True)

    # 3. Process Addition (add_dict)
    process_dict(add_dict, is_subtraction=False)

    # 4. Process Second Subtraction (subtract_dict2)
    process_dict(subtract_dict2, is_subtraction=True)

    # 5. Process Final Total
    x_labels.append("Remaining")
    y_values.append(0)  # Plotly auto-calculates span for total
    measure_types.append("total")
    text_labels.append(f"{running_total:,}")
    raw_counts.append(running_total)

    # 6. Best Fit Logic for Text Positioning (Valid Waterfall Values: 'inside', 'outside', 'auto')
    max_val = max([abs(c) for c in raw_counts] + [1])

    text_positions = []
    text_colors = []

    for count in raw_counts:
        # If the bar height is at least 15% of max value, place text inside the bar
        if (abs(count) / max_val) >= 0.15:
            text_positions.append("inside")
            text_colors.append("white")
        else:
            text_positions.append("outside")
            text_colors.append("black")

    # 7. Build Waterfall Chart
    fig = go.Figure(
        go.Waterfall(
            orientation="v",
            measure=measure_types,
            x=x_labels,
            y=y_values,
            text=text_labels,
            textposition=text_positions,
            textfont=dict(color=text_colors, size=11),
            connector={"line": {"color": "rgb(63, 63, 63)", "width": 1.5}},
            decreasing={"marker": {"color": "#EF553B"}},
            increasing={"marker": {"color": "#00CC96"}},
            totals={"marker": {"color": "#2B5C8F"}},
        )
    )

    # 8. Layout Adjustments
    fig.update_layout(
        title=dict(text=chart_title, x=0.5, font=dict(size=18)),
        showlegend=False,
        template="plotly_white",
        yaxis_title="Count",
        margin=dict(t=80, b=50, l=50, r=50),
    )

    return fig


Plotly_Scatter_Bubble

In [53]:
def plotly_scatter_bubble(
    df: pd.DataFrame,
    x_col: str,
    yaxis: str,
    chartTitle: str,
    exclude_blank: bool = True,
):
    """Creates a Plotly bubble scatter plot using a single category column (e.g., 'PrimaryHealthcare')

    with log-scaled bubble sizes and concatenated X-axis tick labels with total counts.
    """
    df_clean = df.copy()

    # 1. Clean and normalize target string columns
    for col in [x_col, yaxis]:
        if col in df_clean.columns:
            df_clean[col] = (
                df_clean[col]
                .astype(str)
                .str.strip()
                .replace(
                    {
                        "": np.nan,
                        "None": np.nan,
                        "nan": np.nan,
                        "NaN": np.nan,
                        "<NA>": np.nan,
                    }
                )
            )

    # 2. Filter blanks if required
    if exclude_blank:
        df_clean = df_clean.dropna(subset=[x_col, yaxis])

    if df_clean.empty:
        raise ValueError("No valid data remaining after filtering.")

    # 3. Split comma-separated values (e.g., "DM, HT") and explode into individual category rows
    df_clean["X_Category"] = df_clean[x_col].str.split(",")
    exploded = df_clean.explode("X_Category")
    exploded["X_Category"] = exploded["X_Category"].str.strip()

    # Drop any blank categories produced after splitting
    exploded = exploded[
        ~exploded["X_Category"].isin(["", "nan", "NaN", "None", "<NA>"])
        & exploded["X_Category"].notna()
    ]

    # 4. Group and aggregate counts
    grouped = (
        exploded.groupby(["X_Category", yaxis])
        .size()
        .reset_index(name="Count")
    )

    # 5. Calculate total counts and create concatenated X-axis tick labels
    total_counts = (
        grouped.groupby("X_Category")["Count"]
        .sum()
        .reset_index(name="TotalCount")
    )

    total_counts["X_Label"] = total_counts.apply(
        lambda r: f"{r['X_Category']} ({r['TotalCount']})", axis=1
    )

    # Map formatted label back to the grouped dataframe
    category_label_map = dict(
        zip(total_counts["X_Category"], total_counts["X_Label"])
    )
    grouped["X_Label"] = grouped["X_Category"].map(category_label_map)

    # 6. Apply log transformation for bubble sizing
    grouped["LogCount"] = np.log1p(grouped["Count"])

    # 7. Create Bubble Plot using LogCount for size and raw Count for text inside bubbles
    fig = px.scatter(
        grouped,
        x="X_Label",
        y=yaxis,
        size="LogCount",
        color=yaxis,
        size_max=45,
        text="Count",
        title=chartTitle,
        labels={"X_Label": "Category", yaxis: yaxis},
        hover_data={"LogCount": False, "Count": True, "X_Label": True},
    )

    # Display raw count inside bubbles
    fig.update_traces(
        textposition="middle center", textfont=dict(color="white", size=11)
    )

    # 8. Layout adjustments
    fig.update_layout(
        title_x=0.5,
        margin=dict(t=50),
        xaxis=dict(title="Primary Healthcare Category", type="category"),
        yaxis=dict(type="category"),
        showlegend=False,
    )

    return fig

Plotly_Donut

In [54]:
def plot_nested_donut_chart(
    df: pd.DataFrame,
    column_name: str = "PrimaryHealthcare",
    chart_title: str = "Primary Healthcare Category & Overlap Breakdown",
):
    """Generates a multi-level nested donut (Sunburst) chart where the inner ring

    shows every individual primary category, and the outer ring breaks down overlaps.
    """
    df_clean = df.copy()

    # 1. Clean missing/null values
    df_clean[column_name] = (
        df_clean[column_name]
        .astype(str)
        .str.strip()
        .replace(
            {
                "": np.nan,
                "None": np.nan,
                "nan": np.nan,
                "NaN": np.nan,
                "<NA>": np.nan,
            }
        )
    )
    df_clean = df_clean.dropna(subset=[column_name])

    # 2. Preserve full profile (e.g., "DM, HT")
    df_clean["Full_Profile"] = df_clean[column_name]

    # 3. Split comma-separated string and explode into individual primary categories
    df_clean["Primary_Category"] = df_clean[column_name].str.split(",")
    exploded = df_clean.explode("Primary_Category")

    # Clean whitespace around individual categories
    exploded["Primary_Category"] = exploded["Primary_Category"].str.strip()

    # Drop any blank entries produced after splitting
    exploded = exploded[
        ~exploded["Primary_Category"].isin(["", "nan", "NaN", "None", "<NA>"])
        & exploded["Primary_Category"].notna()
    ]

    # 4. Aggregate counts across all primary categories and full profiles
    grouped = (
        exploded.groupby(["Primary_Category", "Full_Profile"])
        .size()
        .reset_index(name="Count")
    )

    # 5. Create Sunburst (Nested Donut)
    fig = px.sunburst(
        grouped,
        path=["Primary_Category", "Full_Profile"],
        values="Count",
        title=chart_title,
        color="Primary_Category",
        color_discrete_sequence=px.colors.qualitative.Pastel,
    )

    fig.update_traces(
        textinfo="label+value+percent entry",
        insidetextorientation="horizontal",
    )

    fig.update_layout(title_x=0.5, margin=dict(t=50, l=0, r=0, b=0))

    return fig

Plotly Combo Chart

In [55]:

def plotly_combo_bar_percent(df: pd.DataFrame, xaxis_str: str, bar_dict: dict, optional_percent_line_list: list = None):

    df = df.copy()

    # Determine unique X-axis categories
    x_categories = sorted(df[xaxis_str].dropna().unique())
    aggregated_counts = {}

    # 1. Filter DataFrame separately per bar criteria & aggregate counts
    for col_name, criteria in bar_dict.items():
        if isinstance(criteria, (list, tuple, set)):
            filtered_df = df[df[col_name].isin(criteria)]
        else:
            filtered_df = df[df[col_name] == criteria]

        counts = filtered_df.groupby(xaxis_str).size()
        aggregated_counts[col_name] = counts.reindex(x_categories, fill_value=0)

    # 2. Setup Figure Layout
    has_secondary = bool(optional_percent_line_list and len(optional_percent_line_list) == 2)

    if has_secondary:
        fig = make_subplots(specs=[[{"secondary_y": True}]])
    else:
        fig = go.Figure()

    # 3. Add Bar Traces (Transformed Y-values via log1p)
    max_count = 0
    for col_name, counts_series in aggregated_counts.items():
        raw_vals = counts_series.values
        max_count = max(max_count, np.max(raw_vals) if len(raw_vals) > 0 else 0)

        # Apply log1p transformation to Y values so 0 stays at 0
        transformed_y = np.log1p(raw_vals)

        trace = go.Bar(
            x=x_categories,
            y=transformed_y,
            name=f"{col_name} ({', '.join(map(str, bar_dict[col_name]))})",
            text=raw_vals,  # Display true raw count as text label on bars
            textposition="inside", #['inside', 'outside', 'auto', 'none']
            hovertemplate="<b>%{x}</b><br>Count: %{text}<extra></extra>",
        )

        if has_secondary:
            fig.add_trace(trace, secondary_y=False)
        else:
            fig.add_trace(trace)

    # 4. Compute & Add Optional Percentage Line on Secondary Y-axis
    pct_max = 100
    if has_secondary:
        num_col, den_col = optional_percent_line_list

        if num_col in aggregated_counts and den_col in aggregated_counts:
            num_series = aggregated_counts[num_col]
            den_series = aggregated_counts[den_col]

            pct_series = np.where(den_series > 0, (num_series / den_series) * 100, 0)
            pct_max = (max(np.max(pct_series), 1) if len(pct_series) > 0 else 100)

            fig.add_trace(
                go.Scatter(
                    x=x_categories,
                    y=pct_series,
                    name=f"% ({num_col} / {den_col})",
                    mode="lines+markers+text",
                    text=[f"<b>{p:.1f}%</b>" for p in pct_series],
                    textposition='top center',
                    textfont=dict(size=12, color="#0C0C0C"),
                    line=dict(dash="4px 4px", width=0.5, color="#0C0C0C"),  # Custom sharp 4px dots
                    marker=dict(size=8,color="#0C0C0C",symbol="diamond",  # Unique marker shape
                                line=dict(width=1.5, color="black"))),
                    secondary_y=True)

    # 5. Build Tick Array for 0, 10, 100, 1000, etc.
    max_power = (int(np.ceil(np.log10(max_count))) if max_count > 0 else 1)  # Power of 10 bounds
    raw_ticks = [0] + [10**i for i in range(1, max_power + 1)]
    transformed_ticks = [np.log1p(t) for t in raw_ticks]
    tick_labels = [str(t) for t in raw_ticks]

    # 6. Apply Layout Configurations
    layout_args = dict(
        title=dict(
            text=f"Number of Cases & Percentage by {xaxis_str}",
            x=0.5,
            xanchor="center",
        ),
        xaxis=dict(title=xaxis_str, type="category"),
        yaxis=dict(
            title="# of Cases",
            type="linear",  # Linear type mapping transformed log values
            tickmode="array",
            tickvals=transformed_ticks,
            ticktext=tick_labels,
            range=[0, np.log1p(max_count * 1.25) if max_count > 0 else 1],
        ),
        barmode="group",
        template="plotly_white",
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5
        ),
        margin=dict(t=100, b=60, l=60, r=60),
    )

    if has_secondary:
        fig.update_layout(**layout_args)
        fig.update_yaxes(
            title_text="Percentage (%)",
            secondary_y=True,
            range=[0, max(10, pct_max * 1.15)],
            showgrid=False,
            ticksuffix="%",
        )
    else:
        fig.update_layout(**layout_args)

    return fig

# plotly_combo_bar_percent(df = df_dashboard, xaxis_str = 'Tsp', bar_dict = {'Reasonforexamination':['Diagnosis'],'Case':['TB']}, optional_percent_line_list = ['Case','Reasonforexamination']).show()

# DASHBOARD

In [ ]:

# ==============================================================================
# 4. DASHBOARD INTERACTION & UI LAYOUT
# ==============================================================================

COLUMNS_SLICER = ['Team','Tsp','Approach','Clinic','Reasonforexamination','Case','Bact_status','Treatmentreferral','MonthDiagnosis11',
                  'Cxrr', 'CXRresult','CXRresult211', 'Genexpertrequested', 'GeneXpertresult','TypeofTBTreatment','TargetCategory']
df_slicer = df_dashboard.copy()
df_slicer["Date"] = pd.to_datetime(df_slicer["Date"])
min_date = pd.to_datetime(df_slicer["Date"].min()).date()
max_date = pd.to_datetime(df_slicer["Date"].max()).date()

def classify_symptomatic(df: pd.DataFrame, symptom_cols, target_val: str = "yes") -> pd.Series:
    cols = [symptom_cols] if isinstance(symptom_cols, str) else list(symptom_cols)
    valid_cols = [c for c in cols if c in df.columns]
    if not valid_cols:
        return pd.Series("Asymptomatic", index=df.index)
    cleaned_symptoms = (
        df[valid_cols]
        .fillna("")
        .astype(str)
        .apply(lambda col: col.str.strip().str.lower())
    )
    has_symptom_mask = cleaned_symptoms.eq(target_val.lower()).any(axis=1)
    symptom_series = pd.Series("Asymptomatic", index=df.index)
    symptom_series[has_symptom_mask] = "Symptomatic"
    return symptom_series
    

def get_options(df, column_name):
    if column_name in df.columns:
        cleaned_values = df[column_name].dropna().astype(str).str.strip()
        unique_vals = sorted(
            list(cleaned_values.replace(["nan", "None", ""], "blank").unique())
        )
        return ["All"] + unique_vals
    return ["All"]

date_from = widgets.DatePicker(description="From:", value=min_date, style={"description_width": "initial"})
date_to = widgets.DatePicker(description="To:", value=max_date, style={"description_width": "initial"})

slicers = {
    col: widgets.SelectMultiple(
        options=get_options(df_slicer,col),
        value=("All",),
        description=f"{col}:",
        layout=widgets.Layout(width="220px", height="100px"),
    )
    for col in COLUMNS_SLICER
}

dashboard_output = widgets.Output()


def update_dashboard(change=None):
    with dashboard_output:
        clear_output(wait=True)

        filtered_df = df_slicer.copy()
        target_df = df_target.copy()
        filtered_df['Symptom'] = classify_symptomatic(filtered_df, COLUMN_SYMPTOM)
        # Date filtering
        if date_from.value:
            filtered_df = filtered_df[filtered_df["Date"].dt.date >= date_from.value]
            target_df = target_df[target_df['ReportingDate'].dt.year >= date_from.value.year]
        if date_to.value:
            filtered_df = filtered_df[filtered_df["Date"].dt.date <= date_to.value]
            target_df = target_df[target_df['ReportingDate'].dt.year <=date_to.value.year]
        # Multi-select filtering
        for col, slicer_widget in slicers.items():
            selected_vals = slicer_widget.value
            if (selected_vals and "All" not in selected_vals and col in filtered_df.columns):
                filtered_df = filtered_df[filtered_df[col].astype(str).str.strip().isin(selected_vals)]
            if (selected_vals and "All" not in selected_vals and col in target_df.columns):
                target_df = target_df[target_df[col].astype(str).str.strip().isin(selected_vals)]

        # #########################################

        achievement = function_indicator_achievement(filtered_df,CRITERIA_INDICATORS)
        progress = function_merge_target(achievement,target_df,indicators=tuple(CRITERIA_INDICATORS.keys()))

        total_attendant = len(filtered_df)


        presumptive_count = achievement['Examined Cases'].sum()
        notified_count = achievement['Notified Cases'].sum()
        bact_confirmed_count = achievement['BC Cases'].sum()

        presumptive_sum_target = progress['Examined Cases Target'].sum()
        notified_sum_target = progress['Notified Cases Target'].sum()
        bact_confirmed_sum_target = progress['BC Cases Target'].sum()

        if date_to.value or date_to.value:
            filtered_progress = progress[progress['ReportingDate'].dt.date.between(date_from.value,date_to.value)]
        
        presumptive_count_target = filtered_progress['Examined Cases Target'].sum()
        notified_count_target = filtered_progress['Notified Cases Target'].sum()
        bact_confirmed_count_target = filtered_progress['BC Cases Target'].sum() 

        presumptive_sub = f"{presumptive_count / presumptive_count_target * 100:.0f}% progress  on {presumptive_count_target:.0f} Targeted <br> ({presumptive_sum_target:.0f} in Total)"
        notified_sub = f"{notified_count / notified_count_target * 100:.0f}% progress on {notified_count_target:.0f} Targeted <br> ({notified_sum_target:.0f} in Total)"
        bact_confirmed_sub = f"{bact_confirmed_count / bact_confirmed_count_target * 100:.0f}% progress on {bact_confirmed_count_target:.0f} Targeted <br> ({bact_confirmed_sum_target:.0f} in Total)"

        # Render KPI Cards HTML
        kpi_html = f"""
        <div style="display: flex; gap: 15px; font-family: Arial, sans-serif; margin-bottom: 20px;">
            <div style="flex: 1; background-color: #f0f4f8; border-left: 5px solid #2b6cb0; padding: 12px; border-radius: 4px;">
                <span style="font-size: 12px; color: #4a5568; font-weight: bold; text-transform: uppercase;">Total Attendant</span>
                <h2 style="margin: 5px 0 0 0; color: #2b6cb0; font-size: 24px;">{total_attendant:,}</h2>
            </div>
            <div style="flex: 1; background-color: #f7fafc; border-left: 5px solid #319795; padding: 12px; border-radius: 4px;">
                <span style="font-size: 12px; color: #4a5568; font-weight: bold; text-transform: uppercase;">Examined Cases</span>
                <h2 style="margin: 5px 0 0 0; color: #319795; font-size: 24px;">{presumptive_count:,}</h2>
                <div style="font-size: 11px; color: #718096; font-weight: bold; margin-top: 6px;">
                    {presumptive_sub}
                </div>
            </div>
            <div style="flex: 1; background-color: #f7fafc; border-left: 5px solid #dd6b20; padding: 12px; border-radius: 4px;">
                <span style="font-size: 12px; color: #4a5568; font-weight: bold; text-transform: uppercase;">Notified Cases</span>
                <h2 style="margin: 5px 0 0 0; color: #dd6b20; font-size: 24px;">{notified_count:,}</h2>
                <div style="font-size: 11px; color: #718096; font-weight: bold; margin-top: 6px;">
                    {notified_sub}
                </div>
            </div>
            <div style="flex: 1; background-color: #f7fafc; border-left: 5px solid #805ad5; padding: 12px; border-radius: 4px;">
                <span style="font-size: 12px; color: #4a5568; font-weight: bold; text-transform: uppercase;">BC Cases</span>
                <h2 style="margin: 5px 0 0 0; color: #805ad5; font-size: 24px;">{bact_confirmed_count:,}</h2>
                <div style="font-size: 11px; color: #718096; font-weight: bold; margin-top: 6px;">
                    {bact_confirmed_sub}
                </div>
            </div>
        </div>
        """

        display(HTML(kpi_html))

        #####################

        # Render Charts

        plotly_achievement_target_dropdown(dataframe = progress,
                achievement_columnList = ["Examined Cases Achievement", "Notified Cases Achievement", "BC Cases Achievement"],
                target_columnList = ["Examined Cases Target","Notified Cases Target","BC Cases Target"],
                period = "Monthly",
                date_col = "ReportingDate").show()

        plotly_variance_heatmap(progress,color_scale_range=(0,200)).show()


        plotly_combo_bar_percent(df = filtered_df, xaxis_str = 'Tsp', bar_dict = {'Reasonforexamination':['Diagnosis'],'Case':['TB']}, 
                                optional_percent_line_list = ['Case','Reasonforexamination']).show()

        plotly_combo_bar_percent(df = filtered_df, xaxis_str = 'Tsp', bar_dict = {'Bact_status':['BC'],'Case':['TB']}, 
                                optional_percent_line_list = ['Bact_status','Case']).show()

        plotly_combo_bar_percent(df = filtered_df, xaxis_str = 'Approach', bar_dict = {'Reasonforexamination':['Diagnosis'],'Case':['TB']}, 
                                optional_percent_line_list = ['Case','Reasonforexamination']).show()

        plotly_combo_bar_percent(df = filtered_df, xaxis_str = 'Approach', bar_dict = {'Bact_status':['BC'],'Case':['TB']}, 
                                optional_percent_line_list = ['Bact_status','Case']).show()
        
        plotly_gender_agegroup(filtered_df,"Sex","Age",500).show()

        # plotly_stack_bar(filtered_df,columns=['TPTregimen','HIVStatus','DM1','TreatmentRegimen','Treatmentreferral','Bact_status'],exclude_blank=True,orientation="h",title="Categorical Distribution").show()
        rename_mapping_stack_bar = {'TPTregimen': 'TPT Provision','HIVStatus': 'HIV Status','DM1': 'DM Status','TreatmentRegimen': 'Treatment Regimen','Treatmentreferral': 'Treatment Registration','Bact_status': 'Bacteriological Status'}
        plotly_stack_bar(filtered_df,columns=list(rename_mapping_stack_bar.keys()),rename_dict=rename_mapping_stack_bar, exclude_blank=True,orientation="h",title="Categorical Breakdown").show()

        # chart_ExaminedCases = widgets.Output()
        # with chart_ExaminedCases:
        #     gender_agegroup = plotly_gender_agegroup(filtered_df,"Sex","Age",500)
        #     display(gender_agegroup)

        # chart_NotifiedCases = widgets.Output()
        # with chart_NotifiedCases:
        #     stack_bar = 
        #     display(stack_bar)

        # summary_row = widgets.HBox(
        #     [chart_ExaminedCases, chart_NotifiedCases],
        #     layout=widgets.Layout(gap="40px", margin="10px 0px 20px 0px"),)
        # display(summary_row)

        plotly_stack_bar(filtered_df,columns=["Treatmentreferral","Tsp","Approach","Case","Sex"],exclude_blank=True,orientation="h",title="Distribution of Cases")

        function_heatmap(filtered_df,"CXRresult","GeneXpertresult").show()

       
        colSankey = {"Team":["MATA","MMA"],"VOL":["Volunteer Referral","Walk-In"],"Referralfor":["CI","Presumptive"],"Case":["TB"],"Bact_status":["BC","CD"],"Treatmentreferral":["Registered"]}
        df_sankey = filtered_df[filtered_df["Reasonforexamination"]=="Diagnosis"]
        function_sankey_cascade_log(dataframe=df_sankey,criteria_dict=colSankey,title="Service Provision Pathway",log_base=10).show()
        # fig.how(
        #     config={
        #         "scrollZoom": True,
        #         "displaylogo": False
        #     }
        # )


        DF_CIDOTS = filtered_df[COLUMN_CI_DOTS]
        DF_CIDOTS = DF_CIDOTS[DF_CIDOTS["Case"] == "TB"]
        DF_CIDOTS = ci_entitled(DF_CIDOTS)

        plotly_waterfall(df=DF_CIDOTS,start_dict={"Notified": {"Case": ["TB"]}},
                         subtract_dict1={"Registered": {"Treatmentreferral": ["Registered"]},"Not Registered": {"Treatmentreferral": ["Not Registered"]}},
                         add_dict={"DS-TB_BC": {"ECI": ["DS-TB_BC"]},"DR-TB": {"ECI": ["DR-TB"]},"TB-HIV": {"ECI": ["TB-HIV"]},"Under5": {"ECI": ["Under5"]}},
                         subtract_dict2={"CI Done": {"ContactInvestigation111": ["Y"]}},
                         chart_title="Contact Investigation Analysis").show()

        plotly_waterfall(df=DF_CIDOTS,start_dict={"Notified": {"Case": ["TB"]}},
                         subtract_dict1={"Volunteer": {"VOL": ["Volunteer Referral"]},"Self": {"VOL": ["Walk-In"]}},
                         add_dict={"Registered": {"Treatmentreferral": ["Registered"]}},
                         subtract_dict2={"DR-TB": {"TypeofTBTreatment": ["DR-TB"]},"DOTS": {"DOTSupervision111": ["Y"]}},
                         chart_title="DOTS Analysis").show()

        plotly_scatter_bubble(df=filtered_df,x_col="PrimaryHealthcare",yaxis="Case",chartTitle="Primary Healthcare Bubble Distribution",exclude_blank=True).show()

        plot_nested_donut_chart(filtered_df, column_name='PrimaryHealthcare').show()

        
        # # plot_clinic_sankey(filtered_df).show()
        # phc = list(df_dashboard['PrimaryHealthcare'].unique())
        # colSankey_phc = {'PublicHealthCare1':['Yes','No'],'TypeofPatient1':['New','Old'],'PrimaryHealthcare':phc}
        # df_sankey_phc = filtered_df.copy()
        # function_sankey_cascade_log(dataframe=df_sankey_phc,criteria_dict=colSankey_phc,title="Service Provision Pathway",log_base=10).show()

        # plot_phc_category_bubble(df_sankey_phc, category_col='PrimaryHealthcare', sep=', ').show()

        print("--- Categorical Summary Table ---")


        charts = plotly_target_achievement_allcharts(
            dataframe=progress,
            date_config={
                "ReportingDate": "Reporting Period"
            },
            bar_configs=[
                {
                    "Examined Cases Target": "Examined Cases Target",
                    "Examined Cases Achievement": "Examined Cases Achievement"
                },
                {
                    "Notified Cases Target": "Notified Cases Target",
                    "Notified Cases Achievement": "Notified Cases Achievement"
                },
                {
                    "BC Cases Target": "BC Cases Target",
                    "BC Cases Achievement": "BC Cases Achievement"
                }
            ],
            optional_percentage=True,
            percentage_calc={
                "Examined Cases": (
                    "Examined Cases Achievement",
                    "Examined Cases Target"
                ),
                "Notified Cases": (
                    "Notified Cases Achievement",
                    "Notified Cases Target"
                ),
                "BC Cases": (
                    "BC Cases Achievement",
                    "BC Cases Target"
                )
            },
            freq="Month"
        )



        chart_ExaminedCases = widgets.Output()
        with chart_ExaminedCases:
            display(charts["Examined Cases"])

        chart_NotifiedCases = widgets.Output()
        with chart_NotifiedCases:
            display(charts["Notified Cases"])

        chart_BCCases = widgets.Output()
        with chart_BCCases:
            display(charts["BC Cases"])

        summary_row = widgets.HBox(
            [chart_ExaminedCases, chart_NotifiedCases,chart_BCCases],
            layout=widgets.Layout(gap="40px", margin="10px 0px 20px 0px"),
        )
        display(summary_row)
       


        print("--- Period Breakdowns ---")

        funnel_column_criteria =  {'Reasonforexamination':['Diagnosis'],
                                'Cxrr':['Requested'],
                                'CXRresult':['TB Suspect','TB Healed','TB Active'],
                                'Genexpertrequested':['Requested'],
                                'GeneXpertresult':['N', 'T', 'TT', 'TI', 'RR',],
                                'Bact_status':['BC'],
                                'Case':['TB'],
                                'Treatmentreferral':['Registered']
                                }
        funnel_column_rename = ['Screening','CXR Request','CXR Abnormality','Gene Request','Gene Result','Bact Confirmed','Notified TB','Treatment Registered']
        plotly_funnel(filtered_df,funnel_column_criteria,funnel_column_rename,'Symptom').show()

### PPTX PREPARATION
        plotly_funnel(filtered_df,funnel_column_criteria,funnel_column_rename,'Approach').show()
        # plotly_table_count_percent(df = filtered_df, column_list = ['Reasonforexamination','Placeforreferral','TypeofTBTreatment','Genexpertrequested','GeneXpertresult', 'Cxrr','CXRresult'], optional_exclude_blank = True,optional_include_total = True).show()

        plotly_table_count_percent(df = filtered_df, column_list = ['TypeofPatient1','Reasonforexamination','TypeofDisease','Transferin','Placeforreferral'], optional_exclude_blank = True,optional_include_total = True).show()
    
        # pivot_outputs = []
        # for col in COLUMNS_TIMELINE:
        #     out = widgets.Output()
        #     with out:
        #         tbl_timeline = generate_period_pivot(filtered_df, col)
        #         display(style_total_rows(tbl_timeline))
        #     pivot_outputs.append(out)

        # pivot_row1 = widgets.HBox(
        #     pivot_outputs[:3],
        #     layout=widgets.Layout(gap="40px", margin="10px 0px"),
        # )
        # pivot_row2 = widgets.HBox(
        #     pivot_outputs[3:],
        #     layout=widgets.Layout(gap="40px", margin="10px 0px"),
        # )
        # display(
        #     widgets.VBox(
        #         [pivot_row1, pivot_row2],
        #         layout=widgets.Layout(margin="0px 0px 20px 0px"),
        #     )
        # )

        # print(f"Showing filtered results ({total_attendant} rows):")
        # if total_attendant > 0:
        #     display(filtered_df)
        # else:
        #     print("No data matches the selected criteria.")


# Event Listeners
date_from.observe(update_dashboard, names="value")
date_to.observe(update_dashboard, names="value")
for slicer_widget in slicers.values():
    slicer_widget.observe(update_dashboard, names="value")

# Final Layout Construction
date_ui = widgets.HBox(
    [date_from, date_to], layout=widgets.Layout(margin="0px 0px 10px 0px")
)
slicers_ui = widgets.HBox(
    list(slicers.values()),
    layout=widgets.Layout(flex_flow="row wrap", gap="15px"),
)

dashboard_layout = widgets.VBox(
    [
        widgets.HTML(
            "<h3 style='font-family:Arial;color:#2d3748;margin-bottom:5px;'>YgnTBPro Data Analysis Dashboard</h3>"
            "<p style='font-size:12px;color:#718096;margin-top:0px;'>Hold Down <b>Ctrl</b> (or <b>Cmd</b> on Mac) to select multiple options.</p>"
        ),
        widgets.VBox([date_ui, slicers_ui]),
        widgets.HTML("<hr style='border: 1px solid #e2e8f0; margin: 15px 0;'>"),
        dashboard_output,
    ]
)



# Display

In [59]:
display(dashboard_layout)
update_dashboard()